In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00000-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00001-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00002-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00003-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00004-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00005-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00006-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00007-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00008-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
/kaggle/input/datasets/akashdogra/cic-iot-2023/part-00009-363d1b

In [2]:

# ============================================================
# O1 - STEP 1: DATASET INVENTORY AND STRUCTURAL INSPECTION
# Thesis: Cross-Dataset Multiclass IoT Intrusion Detection
# ============================================================

import os
import glob
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. DATASET PATHS
# ------------------------------------------------------------

DIAD_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset"

IOT23_PATH = "/kaggle/input/datasets/akashdogra/cic-iot-2023"

OUTPUT_PATH = "/kaggle/working/O1_results"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ------------------------------------------------------------
# 2. VERIFY DATASET DIRECTORIES
# ------------------------------------------------------------

datasets = {
    "CIC_IoT_DIAD_2024": DIAD_PATH,
    "CIC_IoT_2023": IOT23_PATH
}

for name, path in datasets.items():
    print("\n" + "=" * 65)
    print("DATASET:", name)
    print("=" * 65)

    if os.path.exists(path):
        print("Status: AVAILABLE")
    else:
        print("Status: NOT FOUND")
        continue

    files = glob.glob(
        os.path.join(path, "**", "*.csv"),
        recursive=True
    )

    print("Total CSV files:", len(files))

# ------------------------------------------------------------
# 3. INSPECT ALL CSV FILES
# ------------------------------------------------------------

inventory = []

for dataset_name, dataset_path in datasets.items():

    csv_files = glob.glob(
        os.path.join(dataset_path, "**", "*.csv"),
        recursive=True
    )

    for file_path in csv_files:

        file_size_mb = os.path.getsize(file_path) / (1024**2)

        record = {
            "Dataset": dataset_name,
            "File": os.path.basename(file_path),
            "File_Path": file_path,
            "Size_MB": round(file_size_mb, 3),
            "Columns_Count": None,
            "Columns": None,
            "Read_Status": None,
            "Possible_Label_Columns": None,
            "Possible_Time_Columns": None,
            "Possible_ID_Columns": None
        }

        try:
            # Read only five records for structural inspection
            sample = pd.read_csv(
                file_path,
                nrows=5,
                low_memory=False
            )

            columns = sample.columns.tolist()

            record["Columns_Count"] = len(columns)
            record["Columns"] = " | ".join(map(str, columns))
            record["Read_Status"] = "OK"

            # Identify possible label columns
            record["Possible_Label_Columns"] = ", ".join(
                str(c) for c in columns
                if any(k in str(c).lower()
                       for k in ["label", "class", "attack", "category"])
            )

            # Identify possible timestamp columns
            record["Possible_Time_Columns"] = ", ".join(
                str(c) for c in columns
                if any(k in str(c).lower()
                       for k in ["time", "date", "timestamp"])
            )

            # Identify possible device/group identifiers
            record["Possible_ID_Columns"] = ", ".join(
                str(c) for c in columns
                if any(k in str(c).lower()
                       for k in ["device", "source", "capture", "group"])
            )

        except Exception as e:
            record["Read_Status"] = str(e)[:200]

        inventory.append(record)

# ------------------------------------------------------------
# 4. CREATE INVENTORY TABLE
# ------------------------------------------------------------

inventory_df = pd.DataFrame(inventory)

print("\n" + "=" * 65)
print("DATASET INVENTORY SUMMARY")
print("=" * 65)

print(
    inventory_df.groupby("Dataset")
    .agg(
        Total_Files=("File", "count"),
        Total_Size_MB=("Size_MB", "sum"),
        Successful_Reads=(
            "Read_Status",
            lambda x: (x == "OK").sum()
        )
    )
)

# ------------------------------------------------------------
# 5. SAVE RESULTS
# ------------------------------------------------------------

inventory_file = os.path.join(
    OUTPUT_PATH,
    "O1_dataset_inventory.csv"
)

inventory_df.to_csv(inventory_file, index=False)

print("\nInventory saved successfully:")
print(inventory_file)

print("\nFirst 10 inventory records:")
display(inventory_df.head(10))

print("\nO1 STEP 1 COMPLETED")



DATASET: CIC_IoT_DIAD_2024
Status: AVAILABLE
Total CSV files: 126

DATASET: CIC_IoT_2023
Status: AVAILABLE
Total CSV files: 169

DATASET INVENTORY SUMMARY
                   Total_Files  Total_Size_MB  Successful_Reads
Dataset                                                        
CIC_IoT_2023               169      13116.991               169
CIC_IoT_DIAD_2024          126      13585.127               126

Inventory saved successfully:
/kaggle/working/O1_results/O1_dataset_inventory.csv

First 10 inventory records:


,Dataset,File,File_Path,Size_MB,Columns_Count,Columns,Read_Status,Possible_Label_Columns,Possible_Time_Columns,Possible_ID_Columns
0,CIC_IoT_DIAD_2024,BenignTraffic.pcap_Flow.csv,/kaggle/input/datasets/mostafaaliogab/cic-iot-...,94.622,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
1,CIC_IoT_DIAD_2024,BenignTraffic1.pcap_Flow.csv,/kaggle/input/datasets/mostafaaliogab/cic-iot-...,44.293,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
2,CIC_IoT_DIAD_2024,BenignTraffic2.pcap_Flow.csv,/kaggle/input/datasets/mostafaaliogab/cic-iot-...,47.887,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
3,CIC_IoT_DIAD_2024,BenignTraffic3.pcap_Flow.csv,/kaggle/input/datasets/mostafaaliogab/cic-iot-...,20.406,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
4,CIC_IoT_DIAD_2024,DictionaryBruteForce.pcap_Flow.csv,/kaggle/input/datasets/mostafaaliogab/cic-iot-...,1.666,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
5,CIC_IoT_DIAD_2024,DDoS-ACK_Fragmentation.pcap_Flow.csv,/kaggle/input/datasets/mostafaaliogab/cic-iot-...,152.219,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
6,CIC_IoT_DIAD_2024,DDoS-ACK_Fragmentation1.pcap_Flow.csv,/kaggle/input/datasets/mostafaaliogab/cic-iot-...,67.653,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
7,CIC_IoT_DIAD_2024,DDoS-ACK_Fragmentation12.pcap_Flow.csv,/kaggle/input/datasets/mostafaaliogab/cic-iot-...,18.438,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
8,CIC_IoT_DIAD_2024,DDoS-ACK_Fragmentation2.pcap_Flow.csv,/kaggle/input/datasets/mostafaaliogab/cic-iot-...,59.753,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,
9,CIC_IoT_DIAD_2024,DDoS-ACK_Fragmentation3.pcap_Flow.csv,/kaggle/input/datasets/mostafaaliogab/cic-iot-...,84.703,84,Flow ID | Src IP | Src Port | Dst IP | Dst Por...,OK,Label,Timestamp,



O1 STEP 1 COMPLETED


In [3]:

# ============================================================
# O1 - STEP 2: DATA QUALITY ASSESSMENT
# Missing values, infinities, duplicates and data types
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
from collections import defaultdict

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

CHUNK_SIZE = 100_000

datasets = {
    "CIC_IoT_DIAD_2024": DIAD_PATH,
    "CIC_IoT_2023": IOT23_PATH
}

quality_results = []
column_results = []

# ------------------------------------------------------------
# 2. PROCESS EACH CSV FILE IN CHUNKS
# ------------------------------------------------------------

for dataset_name, dataset_path in datasets.items():

    print("\n" + "=" * 70)
    print("DATA QUALITY:", dataset_name)
    print("=" * 70)

    files = glob.glob(
        os.path.join(dataset_path, "**", "*.csv"),
        recursive=True
    )

    for file_number, file_path in enumerate(files, 1):

        print(
            f"\rProcessing {file_number}/{len(files)}: "
            f"{os.path.basename(file_path)[:45]}",
            end=""
        )

        row_count = 0
        missing_counts = defaultdict(int)
        infinity_counts = defaultdict(int)
        dtype_observations = defaultdict(set)
        duplicate_count = 0
        column_names = []
        read_error = None

        try:
            reader = pd.read_csv(
                file_path,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )

            for chunk in reader:

                row_count += len(chunk)

                if not column_names:
                    column_names = chunk.columns.tolist()

                # Missing values
                missing = chunk.isna().sum()

                for col, count in missing.items():
                    missing_counts[col] += int(count)

                # Numerical data types and infinities
                numeric = chunk.select_dtypes(
                    include=[np.number]
                )

                if not numeric.empty:

                    inf_counts = np.isinf(
                        numeric.to_numpy()
                    ).sum(axis=0)

                    for col, count in zip(
                        numeric.columns, inf_counts
                    ):
                        infinity_counts[col] += int(count)

                # Record observed data types
                for col in chunk.columns:
                    dtype_observations[col].add(
                        str(chunk[col].dtype)
                    )

                # Exact duplicate rows within each chunk
                duplicate_count += int(
                    chunk.duplicated().sum()
                )

            # File-level summary
            quality_results.append({
                "Dataset": dataset_name,
                "File": os.path.basename(file_path),
                "Rows": row_count,
                "Columns": len(column_names),
                "Missing_Values": sum(missing_counts.values()),
                "Infinite_Values": sum(infinity_counts.values()),
                "Within_Chunk_Duplicates": duplicate_count,
                "Read_Status": "OK"
            })

            # Column-level summary
            for col in column_names:

                column_results.append({
                    "Dataset": dataset_name,
                    "File": os.path.basename(file_path),
                    "Column": col,
                    "Missing_Count": missing_counts[col],
                    "Missing_Percent": (
                        missing_counts[col] / row_count * 100
                        if row_count else 0
                    ),
                    "Infinite_Count": infinity_counts[col],
                    "Observed_Dtypes": ", ".join(
                        sorted(dtype_observations[col])
                    )
                })

        except Exception as e:

            read_error = str(e)[:300]

            quality_results.append({
                "Dataset": dataset_name,
                "File": os.path.basename(file_path),
                "Rows": row_count,
                "Columns": len(column_names),
                "Missing_Values": None,
                "Infinite_Values": None,
                "Within_Chunk_Duplicates": None,
                "Read_Status": read_error
            })

    print("\nFinished:", dataset_name)

# ------------------------------------------------------------
# 3. CREATE RESULTS TABLES
# ------------------------------------------------------------

quality_df = pd.DataFrame(quality_results)
columns_quality_df = pd.DataFrame(column_results)

print("\n" + "=" * 70)
print("DATA QUALITY SUMMARY")
print("=" * 70)

summary = quality_df.groupby("Dataset").agg(
    Total_Rows=("Rows", "sum"),
    Total_Files=("File", "count"),
    Total_Missing=("Missing_Values", "sum"),
    Total_Infinite=("Infinite_Values", "sum"),
    Within_Chunk_Duplicates=(
        "Within_Chunk_Duplicates", "sum"
    )
)

display(summary)

# ------------------------------------------------------------
# 4. TOP MISSING-VALUE FEATURES
# ------------------------------------------------------------

print("\nTOP 20 COLUMNS WITH MISSING VALUES")

missing_summary = (
    columns_quality_df.groupby(
        ["Dataset", "Column"]
    )["Missing_Count"]
    .sum()
    .reset_index()
    .sort_values(
        "Missing_Count",
        ascending=False
    )
)

display(missing_summary.head(20))

# ------------------------------------------------------------
# 5. SAVE RESULTS
# ------------------------------------------------------------

quality_df.to_csv(
    os.path.join(
        OUTPUT_PATH,
        "O1_data_quality_summary.csv"
    ),
    index=False
)

columns_quality_df.to_csv(
    os.path.join(
        OUTPUT_PATH,
        "O1_column_quality_details.csv"
    ),
    index=False
)

missing_summary.to_csv(
    os.path.join(
        OUTPUT_PATH,
        "O1_missing_value_summary.csv"
    ),
    index=False
)

print("\nAll quality reports saved in:", OUTPUT_PATH)
print("O1 STEP 2 COMPLETED")



DATA QUALITY: CIC_IoT_DIAD_2024
Processing 126/126: SqlInjection.pcap_Flow.csv
Finished: CIC_IoT_DIAD_2024

DATA QUALITY: CIC_IoT_2023
Processing 169/169: part-00168-363d1ba3-8ab5-4f96-bc25-4d5862db7c
Finished: CIC_IoT_2023

DATA QUALITY SUMMARY


,Total_Rows,Total_Files,Total_Missing,Total_Infinite,Within_Chunk_Duplicates
Dataset,,,,,
CIC_IoT_2023,46686579,169,0,0,0
CIC_IoT_DIAD_2024,26969077,126,86310,93042,33012



TOP 20 COLUMNS WITH MISSING VALUES


,Dataset,Column,Missing_Count
281,CIC_IoT_DIAD_2024,Flow Bytes/s,86310
228,CIC_IoT_DIAD_2024,58.68986283848344,0
227,CIC_IoT_DIAD_2024,57126.0.2,0
226,CIC_IoT_DIAD_2024,57126.0.1,0
225,CIC_IoT_DIAD_2024,57126.0,0
224,CIC_IoT_DIAD_2024,57126,0
223,CIC_IoT_DIAD_2024,55338,0
222,CIC_IoT_DIAD_2024,532845.5,0
221,CIC_IoT_DIAD_2024,51.1,0
209,CIC_IoT_DIAD_2024,416190.5.1,0



All quality reports saved in: /kaggle/working/O1_results
O1 STEP 2 COMPLETED


In [4]:
# ============================================================
# O1 STEP 3: DIAD CSV HEADER CONSISTENCY AUDIT
# ============================================================

import os
import glob
import csv
import re
import pandas as pd
from collections import Counter, defaultdict

DIAD_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset"

files = glob.glob(os.path.join(DIAD_PATH, "**", "*.csv"), recursive=True)

# Common expected traffic-feature/header terms
known_terms = [
    "label", "timestamp", "flow id", "src ip", "dst ip",
    "src port", "dst port", "protocol", "flow duration",
    "flow bytes", "flow packets", "packet length",
    "fwd", "bwd", "total length", "header length",
    "flow packets/s", "flow bytes/s"
]

def read_first_line(path):
    """Read the first physical line of a CSV file."""
    try:
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            return f.readline().strip()
    except Exception as e:
        return f"READ_ERROR: {e}"

def assess_header(path):
    first_line = read_first_line(path)

    try:
        columns = next(csv.reader([first_line]))
    except Exception:
        columns = []

    normalized = [str(c).strip().lower() for c in columns]
    joined = " ".join(normalized)

    matched_terms = [term for term in known_terms if term in joined]

    # A header is more likely when recognizable feature names are present.
    # This is a screening rule, not a definitive proof.
    likely_header = len(matched_terms) >= 2

    return {
        "File": os.path.basename(path),
        "Column_Count": len(columns),
        "Likely_Header": likely_header,
        "Recognized_Header_Terms": ", ".join(matched_terms),
        "First_Line_Sample": first_line[:250]
    }

audit_results = [assess_header(path) for path in files]
header_audit = pd.DataFrame(audit_results)

print("TOTAL DIAD FILES:", len(header_audit))
print("\nHEADER CLASSIFICATION:")
print(header_audit["Likely_Header"].value_counts(dropna=False))

print("\nFILES THAT MAY HAVE NO HEADER:")
display(
    header_audit[header_audit["Likely_Header"] == False][
        ["File", "Column_Count", "First_Line_Sample"]
    ].head(30)
)

print("\nFILES WITH LIKELY HEADERS:")
display(
    header_audit[header_audit["Likely_Header"] == True][
        ["File", "Column_Count", "Recognized_Header_Terms"]
    ].head(15)
)

# Save audit
audit_path = "/kaggle/working/O1_results/O1_DIAD_header_audit.csv"
header_audit.to_csv(audit_path, index=False)

print("\nHeader audit saved to:", audit_path)

TOTAL DIAD FILES: 126

HEADER CLASSIFICATION:
Likely_Header
True     121
False      5
Name: count, dtype: int64

FILES THAT MAY HAVE NO HEADER:


,File,Column_Count,First_Line_Sample
70,DoS-TCP_Flood.pcap_Flow.csv,84,"35.185.101.66-192.168.137.72-443-33777-6,35.18..."
71,DoS-TCP_Flood1.pcap_Flow.csv,84,"192.168.137.164-192.168.137.39-80-34050-6,192...."
72,DoS-TCP_Flood10.pcap_Flow.csv,84,"44.231.176.168-192.168.137.19-443-47717-6,44.2..."
73,DoS-TCP_Flood2.pcap_Flow.csv,84,"192.168.137.31-172.253.115.95-50439-443-6,192...."
74,DoS-TCP_Flood3.pcap_Flow.csv,84,"142.251.41.67-192.168.137.93-80-55338-6,142.25..."



FILES WITH LIKELY HEADERS:


,File,Column_Count,Recognized_Header_Terms
0,BenignTraffic.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
1,BenignTraffic1.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
2,BenignTraffic2.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
3,BenignTraffic3.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
4,DictionaryBruteForce.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
5,DDoS-ACK_Fragmentation.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
6,DDoS-ACK_Fragmentation1.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
7,DDoS-ACK_Fragmentation12.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
8,DDoS-ACK_Fragmentation2.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."
9,DDoS-ACK_Fragmentation3.pcap_Flow.csv,84,"label, timestamp, flow id, src ip, dst ip, src..."



Header audit saved to: /kaggle/working/O1_results/O1_DIAD_header_audit.csv


In [5]:
# ============================================================
# O1 STEP 3A: FIX FILE PATHS AND VERIFY DIAD SCHEMA
# ============================================================

import os
import glob
import csv
import pandas as pd

DIAD_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset"
OUTPUT_PATH = "/kaggle/working/O1_results"

# Find actual CSV paths recursively
all_files = glob.glob(
    os.path.join(DIAD_PATH, "**", "*.csv"),
    recursive=True
)

# Map filenames to actual paths
file_path_map = {
    os.path.basename(path): path
    for path in all_files
}

audit = pd.read_csv(
    os.path.join(OUTPUT_PATH, "O1_DIAD_header_audit.csv")
)

headerless_files = audit.loc[
    audit["Likely_Header"] == False, "File"
].tolist()

headered_files = audit.loc[
    audit["Likely_Header"] == True, "File"
].tolist()

print("Actual CSV files found:", len(all_files))

# Find a reference file that exists
reference_name = next(
    (name for name in headered_files if name in file_path_map),
    None
)

if reference_name is None:
    raise FileNotFoundError("No reference headered file was found.")

reference_path = file_path_map[reference_name]

reference_columns = pd.read_csv(
    reference_path,
    nrows=0
).columns.tolist()

print("\nREFERENCE FILE:", reference_name)
print("REFERENCE COLUMN COUNT:", len(reference_columns))

# Check all headered files
schema_mismatches = []

for filename in headered_files:
    if filename not in file_path_map:
        schema_mismatches.append(filename)
        continue

    columns = pd.read_csv(
        file_path_map[filename],
        nrows=0
    ).columns.tolist()

    if columns != reference_columns:
        schema_mismatches.append(filename)

print("\nHEADERED FILES:", len(headered_files))
print("SCHEMA MISMATCHES:", len(schema_mismatches))
print("MISMATCHED FILES:", schema_mismatches)

# Check headerless files
results = []

for filename in headerless_files:
    if filename not in file_path_map:
        results.append({
            "File": filename,
            "Field_Count": None,
            "Matches_Reference": False,
            "Status": "FILE NOT FOUND"
        })
        continue

    with open(
        file_path_map[filename],
        "r",
        encoding="utf-8",
        errors="replace"
    ) as f:
        first_line = f.readline()

    field_count = len(next(csv.reader([first_line])))

    results.append({
        "File": filename,
        "Field_Count": field_count,
        "Matches_Reference": field_count == len(reference_columns),
        "Status": "OK" if field_count == len(reference_columns)
                  else "FIELD COUNT MISMATCH"
    })

print("\nHEADERLESS FILE VERIFICATION:")
display(pd.DataFrame(results))

print("\nREFERENCE COLUMN NAMES:")
print(reference_columns)

Actual CSV files found: 126

REFERENCE FILE: BenignTraffic.pcap_Flow.csv
REFERENCE COLUMN COUNT: 84

HEADERED FILES: 121
SCHEMA MISMATCHES: 0
MISMATCHED FILES: []

HEADERLESS FILE VERIFICATION:


,File,Field_Count,Matches_Reference,Status
0,DoS-TCP_Flood.pcap_Flow.csv,84,True,OK
1,DoS-TCP_Flood1.pcap_Flow.csv,84,True,OK
2,DoS-TCP_Flood10.pcap_Flow.csv,84,True,OK
3,DoS-TCP_Flood2.pcap_Flow.csv,84,True,OK
4,DoS-TCP_Flood3.pcap_Flow.csv,84,True,OK



REFERENCE COLUMN NAMES:
['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag 

In [6]:
# ============================================================
# O1 STEP 3B: CORRECTED DIAD DATA-QUALITY SCAN
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

DIAD_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset"
OUTPUT_PATH = "/kaggle/working/O1_results"
CHUNK_SIZE = 100_000

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Locate actual files
files = glob.glob(
    os.path.join(DIAD_PATH, "**", "*.csv"),
    recursive=True
)

file_map = {os.path.basename(p): p for p in files}

# Load verified header audit
audit = pd.read_csv(
    os.path.join(OUTPUT_PATH, "O1_DIAD_header_audit.csv")
)

headerless_files = set(
    audit.loc[audit["Likely_Header"] == False, "File"]
)

# Obtain the verified 84-column schema
reference_file = next(
    name for name in audit.loc[
        audit["Likely_Header"] == True, "File"
    ]
    if name in file_map
)

reference_columns = pd.read_csv(
    file_map[reference_file], nrows=0
).columns.tolist()

print("Verified schema columns:", len(reference_columns))
print("Headerless files:", len(headerless_files))

total_rows = 0
total_missing = 0
total_infinite = 0
total_duplicates = 0
file_results = []
column_missing = {}
read_errors = []

for file_number, (filename, path) in enumerate(file_map.items(), 1):

    print(f"Processing {file_number}/{len(file_map)}: {filename}")

    file_rows = 0
    file_missing = 0
    file_infinite = 0
    file_duplicates = 0

    try:
        if filename in headerless_files:
            reader = pd.read_csv(
                path,
                header=None,
                names=reference_columns,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )
        else:
            reader = pd.read_csv(
                path,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )

        for chunk in reader:

            file_rows += len(chunk)

            # Missing values
            missing_counts = chunk.isna().sum()

            file_missing += int(missing_counts.sum())

            for col, count in missing_counts.items():
                column_missing[col] = (
                    column_missing.get(col, 0) + int(count)
                )

            # Infinite values in numeric columns
            numeric = chunk.select_dtypes(include=[np.number])

            if not numeric.empty:
                values = numeric.to_numpy(dtype=float)
                infinite_mask = np.isinf(values)

                file_infinite += int(infinite_mask.sum())

            # Duplicates within each chunk only
            file_duplicates += int(
                chunk.duplicated().sum()
            )

        total_rows += file_rows
        total_missing += file_missing
        total_infinite += file_infinite
        total_duplicates += file_duplicates

        file_results.append({
            "File": filename,
            "Rows": file_rows,
            "Missing": file_missing,
            "Infinite": file_infinite,
            "Within_Chunk_Duplicates": file_duplicates,
            "Status": "Success"
        })

    except Exception as e:
        read_errors.append({
            "File": filename,
            "Error": str(e)
        })

        file_results.append({
            "File": filename,
            "Rows": file_rows,
            "Missing": file_missing,
            "Infinite": file_infinite,
            "Within_Chunk_Duplicates": file_duplicates,
            "Status": f"Error: {str(e)[:150]}"
        })

# Summary
summary = pd.DataFrame([{
    "Dataset": "CIC_IoT_DIAD_2024",
    "Total_Rows": total_rows,
    "Total_Files": len(files),
    "Total_Missing": total_missing,
    "Total_Infinite": total_infinite,
    "Within_Chunk_Duplicates": total_duplicates,
    "Read_Errors": len(read_errors)
}])

file_report = pd.DataFrame(file_results)

missing_report = pd.DataFrame([
    {"Column": col, "Missing_Count": count}
    for col, count in column_missing.items()
    if count > 0
]).sort_values("Missing_Count", ascending=False)

# Save reports
summary.to_csv(
    os.path.join(OUTPUT_PATH, "O1_DIAD_corrected_quality_summary.csv"),
    index=False
)

file_report.to_csv(
    os.path.join(OUTPUT_PATH, "O1_DIAD_corrected_file_quality.csv"),
    index=False
)

missing_report.to_csv(
    os.path.join(OUTPUT_PATH, "O1_DIAD_corrected_missing_values.csv"),
    index=False
)

print("\n========== CORRECTED DIAD QUALITY SUMMARY ==========")
display(summary)

print("\n========== MISSING VALUE COLUMNS ==========")
display(missing_report)

print("\n========== FILE PROCESSING STATUS ==========")
print(file_report["Status"].value_counts())

if read_errors:
    print("\n========== READ ERRORS ==========")
    display(pd.DataFrame(read_errors))

print("\nReports saved in:", OUTPUT_PATH)

Verified schema columns: 84
Headerless files: 5
Processing 1/126: BenignTraffic.pcap_Flow.csv
Processing 2/126: BenignTraffic1.pcap_Flow.csv
Processing 3/126: BenignTraffic2.pcap_Flow.csv
Processing 4/126: BenignTraffic3.pcap_Flow.csv
Processing 5/126: DictionaryBruteForce.pcap_Flow.csv
Processing 6/126: DDoS-ACK_Fragmentation.pcap_Flow.csv
Processing 7/126: DDoS-ACK_Fragmentation1.pcap_Flow.csv
Processing 8/126: DDoS-ACK_Fragmentation12.pcap_Flow.csv
Processing 9/126: DDoS-ACK_Fragmentation2.pcap_Flow.csv
Processing 10/126: DDoS-ACK_Fragmentation3.pcap_Flow.csv
Processing 11/126: DDoS-ACK_Fragmentation4.pcap_Flow.csv
Processing 12/126: DDoS-ACK_Fragmentation5.pcap_Flow.csv
Processing 13/126: DDoS-ACK_Fragmentation6.pcap_Flow.csv
Processing 14/126: DDoS-ACK_Fragmentation7.pcap_Flow.csv
Processing 15/126: DDoS-ACK_Fragmentation8.pcap_Flow.csv
Processing 16/126: DDoS-ICMP_Flood.pcap_Flow.csv
Processing 17/126: DDoS-ICMP_Flood1.pcap_Flow.csv
Processing 18/126: DDoS-ICMP_Flood10.pcap_Flow.

,Dataset,Total_Rows,Total_Files,Total_Missing,Total_Infinite,Within_Chunk_Duplicates,Read_Errors
0,CIC_IoT_DIAD_2024,26969082,126,86310,93042,33012,0



========== MISSING VALUE COLUMNS ==========


,Column,Missing_Count
0,Flow Bytes/s,86310



========== FILE PROCESSING STATUS ==========
Status
Success    126
Name: count, dtype: int64

Reports saved in: /kaggle/working/O1_results


In [7]:
# ============================================================
# O1 STEP 4A: DIAD CLASS DISTRIBUTION AUDIT
# ============================================================

import os
import glob
import pandas as pd

DIAD_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset"
OUTPUT_PATH = "/kaggle/working/O1_results"
CHUNK_SIZE = 100_000

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Locate all DIAD CSV files
files = glob.glob(
    os.path.join(DIAD_PATH, "**", "*.csv"),
    recursive=True
)

file_map = {os.path.basename(p): p for p in files}

# Load verified header audit
audit = pd.read_csv(
    os.path.join(OUTPUT_PATH, "O1_DIAD_header_audit.csv")
)

headerless_files = set(
    audit.loc[audit["Likely_Header"] == False, "File"]
)

# Get the verified column names
reference_file = next(
    name for name in audit.loc[
        audit["Likely_Header"] == True, "File"
    ]
    if name in file_map
)

reference_columns = pd.read_csv(
    file_map[reference_file],
    nrows=0
).columns.tolist()

if "Label" not in reference_columns:
    raise ValueError("Label column not found in DIAD reference schema.")

# Count actual labels across all files
overall_counts = {}
file_label_counts = []
total_records = 0
missing_labels = 0
read_errors = []

for number, (filename, path) in enumerate(file_map.items(), 1):

    print(f"Scanning {number}/{len(file_map)}: {filename}")

    try:
        if filename in headerless_files:
            reader = pd.read_csv(
                path,
                header=None,
                names=reference_columns,
                usecols=["Label"],
                chunksize=CHUNK_SIZE
            )
        else:
            reader = pd.read_csv(
                path,
                usecols=["Label"],
                chunksize=CHUNK_SIZE
            )

        current_file_counts = {}

        for chunk in reader:
            labels = chunk["Label"]

            missing_labels += int(labels.isna().sum())
            total_records += len(labels)

            # Preserve actual label text, trimming surrounding spaces
            labels = labels.astype("string").str.strip()
            counts = labels.value_counts(dropna=False)

            for label, count in counts.items():
                label_name = (
                    "<MISSING>" if pd.isna(label) else str(label)
                )

                overall_counts[label_name] = (
                    overall_counts.get(label_name, 0) + int(count)
                )

                current_file_counts[label_name] = (
                    current_file_counts.get(label_name, 0) + int(count)
                )

        for label_name, count in current_file_counts.items():
            file_label_counts.append({
                "File": filename,
                "Label": label_name,
                "Count": count
            })

    except Exception as e:
        read_errors.append({
            "File": filename,
            "Error": str(e)
        })

# Overall class distribution
class_distribution = pd.DataFrame([
    {"Label": label, "Count": count}
    for label, count in overall_counts.items()
])

class_distribution = class_distribution.sort_values(
    "Count", ascending=False
).reset_index(drop=True)

class_distribution["Percentage"] = (
    class_distribution["Count"] / total_records * 100
)

# Per-file label distribution
file_distribution = pd.DataFrame(file_label_counts)

# Imbalance ratio, excluding missing labels
valid_counts = [
    count for label, count in overall_counts.items()
    if label != "<MISSING>" and count > 0
]

if valid_counts:
    imbalance_ratio = max(valid_counts) / min(valid_counts)
else:
    imbalance_ratio = None

# Save reports
class_distribution.to_csv(
    os.path.join(OUTPUT_PATH, "O1_DIAD_class_distribution.csv"),
    index=False
)

file_distribution.to_csv(
    os.path.join(OUTPUT_PATH, "O1_DIAD_file_label_distribution.csv"),
    index=False
)

# Display results
print("\n========== DIAD CLASS DISTRIBUTION ==========")
display(class_distribution)

print("\nTotal records scanned:", total_records)
print("Missing labels:", missing_labels)
print("Number of observed label categories:", len(overall_counts))
print("Class imbalance ratio (largest/smallest non-missing class):",
      imbalance_ratio)

print("\nRead errors:", len(read_errors))

if read_errors:
    display(pd.DataFrame(read_errors))

print("\nReports saved in:", OUTPUT_PATH)

Scanning 1/126: BenignTraffic.pcap_Flow.csv
Scanning 2/126: BenignTraffic1.pcap_Flow.csv
Scanning 3/126: BenignTraffic2.pcap_Flow.csv
Scanning 4/126: BenignTraffic3.pcap_Flow.csv
Scanning 5/126: DictionaryBruteForce.pcap_Flow.csv
Scanning 6/126: DDoS-ACK_Fragmentation.pcap_Flow.csv
Scanning 7/126: DDoS-ACK_Fragmentation1.pcap_Flow.csv
Scanning 8/126: DDoS-ACK_Fragmentation12.pcap_Flow.csv
Scanning 9/126: DDoS-ACK_Fragmentation2.pcap_Flow.csv
Scanning 10/126: DDoS-ACK_Fragmentation3.pcap_Flow.csv
Scanning 11/126: DDoS-ACK_Fragmentation4.pcap_Flow.csv
Scanning 12/126: DDoS-ACK_Fragmentation5.pcap_Flow.csv
Scanning 13/126: DDoS-ACK_Fragmentation6.pcap_Flow.csv
Scanning 14/126: DDoS-ACK_Fragmentation7.pcap_Flow.csv
Scanning 15/126: DDoS-ACK_Fragmentation8.pcap_Flow.csv
Scanning 16/126: DDoS-ICMP_Flood.pcap_Flow.csv
Scanning 17/126: DDoS-ICMP_Flood1.pcap_Flow.csv
Scanning 18/126: DDoS-ICMP_Flood10.pcap_Flow.csv
Scanning 19/126: DDoS-ICMP_Flood11.pcap_Flow.csv
Scanning 20/126: DDoS-ICMP_Floo

,Label,Count,Percentage
0,NeedManualLabel,26969082,100.0



Total records scanned: 26969082
Missing labels: 0
Number of observed label categories: 1
Class imbalance ratio (largest/smallest non-missing class): 1.0

Read errors: 0

Reports saved in: /kaggle/working/O1_results


In [8]:
# ============================================================
# O1 STEP 4B: INVESTIGATE DIAD LABEL SOURCE
# ============================================================

import os
import glob
import pandas as pd

DIAD_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset"

# 1. Find README and documentation files
documentation_files = []

for root, dirs, files in os.walk(DIAD_PATH):
    for filename in files:
        if filename.lower().startswith("readme") or filename.lower().endswith(
            (".txt", ".md")
        ):
            documentation_files.append(os.path.join(root, filename))

print("DOCUMENTATION FILES FOUND:")
for path in documentation_files:
    print(path)

# 2. Display the first 10 lines of each README/text file
for path in documentation_files:
    print("\nDOCUMENT:", path)

    try:
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            for line_number, line in enumerate(f):
                if line_number >= 10:
                    break
                print(line.rstrip())
    except Exception as e:
        print("Could not read:", e)

# 3. Inspect labels in representative files
sample_files = [
    "BenignTraffic.pcap_Flow\\.csv",
    "DictionaryBruteForce.pcap_Flow\\.csv",
    "DDoS-ICMP_Flood.pcap_Flow\\.csv",
    "Mirai-greeth_flood.pcap_Flow\\.csv",
    "MITM-ArpSpoofing.pcap_Flow\\.csv"
]

all_csv = glob.glob(
    os.path.join(DIAD_PATH, "**", "*.csv"),
    recursive=True
)

file_map = {os.path.basename(p): p for p in all_csv}

print("\nREPRESENTATIVE FILE LABEL INSPECTION:")

for filename in sample_files:
    path = file_map.get(filename)

    if path is None:
        print("\nFILE NOT FOUND:", filename)
        continue

    df = pd.read_csv(path, usecols=["Label"], nrows=20)

    print("\nFILE:", filename)
    print("LABEL VALUES:")
    print(df["Label"].value_counts(dropna=False).to_dict())
    print("SAMPLE LABELS:")
    print(df["Label"].head(5).tolist())

DOCUMENTATION FILES FOUND:
/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024/README.txt.txt

DOCUMENT: /kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024/README.txt.txt
The CICFlowMeter is an open source tool that generates Biflows from pcap files, and extracts features from these flows.



NOTE: TCP flows are usually terminated upon connection teardown (by FIN packet) while UDP flows are terminated by a flow timeout. The flow timeout value can be assigned arbitrarily by the individual scheme e.g., 600 seconds for both TCP and UDP.


NOTE:

The data found in this folder is comprised of 8 folders(Benign + 7 Attacks). Each folder will contain csv files for the kind of traffic specified by the folder name.

REPRESENTATIVE FILE LABEL INSPECTION:

FILE NOT FOUND: BenignTraffic.pcap_Flow\.csv

FILE NOT FOUND: DictionaryBruteForce.pcap_Flow\.csv

FILE NOT FOUND: DDoS-ICMP_Flood.pcap_Flow\.csv

FILE NOT FOUND: Mirai-greeth_flood.pcap

In [9]:
import os

DIAD_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset"

print("ACTUAL DIAD CSV FILENAMES")
print("=" * 80)

csv_files = []

for root, dirs, files in os.walk(DIAD_PATH):
    for filename in files:
        if filename.lower().endswith(".csv"):
            full_path = os.path.join(root, filename)
            csv_files.append(full_path)

print("Total CSV files:", len(csv_files))

keywords = [
    "BenignTraffic",
    "DictionaryBruteForce",
    "DDoS-ICMP_Flood",
    "Mirai-greeth_flood",
    "MITM-ArpSpoofing"
]

for keyword in keywords:
    print(f"\nSearching for: {keyword}")

    matches = [
        path for path in csv_files
        if keyword.lower() in os.path.basename(path).lower()
    ]

    if matches:
        for path in matches[:5]:
            print("Filename:", repr(os.path.basename(path)))
            print("Full path:", path)
    else:
        print("No matching filename found.")

print("\nFIRST 15 ACTUAL CSV FILENAMES")
for path in csv_files[:15]:
    print(repr(os.path.basename(path)))

ACTUAL DIAD CSV FILENAMES
Total CSV files: 126

Searching for: BenignTraffic
Filename: 'BenignTraffic.pcap_Flow.csv'
Full path: /kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024/Benign/BenignTraffic.pcap_Flow.csv
Filename: 'BenignTraffic1.pcap_Flow.csv'
Full path: /kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024/Benign/BenignTraffic1.pcap_Flow.csv
Filename: 'BenignTraffic2.pcap_Flow.csv'
Full path: /kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024/Benign/BenignTraffic2.pcap_Flow.csv
Filename: 'BenignTraffic3.pcap_Flow.csv'
Full path: /kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024/Benign/BenignTraffic3.pcap_Flow.csv

Searching for: DictionaryBruteForce
Filename: 'DictionaryBruteForce.pcap_Flow.csv'
Full path: /kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024/BruteForce/DictionaryBruteForce.pcap_Flow.csv

Searching for: DDoS-ICMP_F

In [10]:
import os
import pandas as pd

DIAD_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset"

sample_names = [
    "BenignTraffic.pcap_Flow.csv",
    "DictionaryBruteForce.pcap_Flow.csv",
    "DDoS-ICMP_Flood.pcap_Flow.csv",
    "Mirai-greeth_flood.pcap_Flow.csv",
    "MITM-ArpSpoofing.pcap_Flow.csv"
]

# Find actual paths using filenames
file_map = {}

for root, dirs, files in os.walk(DIAD_PATH):
    for filename in files:
        if filename in sample_names:
            file_map[filename] = os.path.join(root, filename)

print("ACTUAL LABEL INSPECTION")
print("=" * 70)

for filename in sample_names:

    path = file_map.get(filename)

    if path is None:
        print(f"\nNOT FOUND: {filename}")
        continue

    print(f"\nFILE: {filename}")
    print("FOLDER:", os.path.basename(os.path.dirname(path)))

    # Read the actual Label column
    df = pd.read_csv(path, usecols=["Label"], low_memory=False)

    print("Rows:", len(df))
    print("Unique labels:", df["Label"].nunique())

    print("\nLabel distribution:")
    print(df["Label"].value_counts(dropna=False).head(20))

    print("\nFirst 10 label values:")
    print(df["Label"].head(10).tolist())

print("\nINSPECTION COMPLETED")

ACTUAL LABEL INSPECTION

FILE: BenignTraffic.pcap_Flow.csv
FOLDER: Benign
Rows: 183630
Unique labels: 1

Label distribution:
Label
NeedManualLabel    183630
Name: count, dtype: int64

First 10 label values:
['NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel']

FILE: DictionaryBruteForce.pcap_Flow.csv
FOLDER: BruteForce
Rows: 3619
Unique labels: 1

Label distribution:
Label
NeedManualLabel    3619
Name: count, dtype: int64

First 10 label values:
['NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel', 'NeedManualLabel']

FILE: DDoS-ICMP_Flood.pcap_Flow.csv
FOLDER: DDoS ICMP Flood
Rows: 10415
Unique labels: 1

Label distribution:
Label
NeedManualLabel    10415
Name: count, dtype: int64

First 10 label values:
['NeedManualLabel', 'NeedManu

In [11]:
import os
import pandas as pd

DIAD_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset"


folder_files = {}

for root, dirs, files in os.walk(DIAD_PATH):
    for filename in files:
        if filename.lower().endswith(".csv"):
            folder = os.path.basename(root)
            folder_files.setdefault(folder, []).append(filename)

print("DIAD FOLDER AUDIT")
print("=" * 70)

for folder, files in sorted(folder_files.items()):
    print(f"\nFolder: {folder}")
    print("CSV files:", len(files))
    print("Examples:")
    for filename in files[:5]:
        print("  ", filename)

print("\nTOTAL CSV FILES:", sum(len(files) for files in folder_files.values()))

# Search for documentation files
print("\nDOCUMENTATION FILE SEARCH")
print("=" * 70)

documentation = []

for root, dirs, files in os.walk(DIAD_PATH):
    for filename in files:
        if filename.lower() in [
            "readme.txt", "readme.md", "readme",
            "description.txt", "metadata.txt"
        ]:
            documentation.append(os.path.join(root, filename))

if documentation:
    for path in documentation:
        print(path)
else:
    print("No README or metadata files found.")

DIAD FOLDER AUDIT

Folder: ARP Spoofing
CSV files: 2
Examples:
   MITM-ArpSpoofing.pcap_Flow.csv
   MITM-ArpSpoofing1.pcap_Flow.csv

Folder: Benign
CSV files: 4
Examples:
   BenignTraffic.pcap_Flow.csv
   BenignTraffic1.pcap_Flow.csv
   BenignTraffic2.pcap_Flow.csv
   BenignTraffic3.pcap_Flow.csv

Folder: BruteForce
CSV files: 1
Examples:
   DictionaryBruteForce.pcap_Flow.csv

Folder: DDoS ACK Fragmentation
CSV files: 10
Examples:
   DDoS-ACK_Fragmentation.pcap_Flow.csv
   DDoS-ACK_Fragmentation1.pcap_Flow.csv
   DDoS-ACK_Fragmentation12.pcap_Flow.csv
   DDoS-ACK_Fragmentation2.pcap_Flow.csv
   DDoS-ACK_Fragmentation3.pcap_Flow.csv

Folder: DDoS ICMP Flood
CSV files: 27
Examples:
   DDoS-ICMP_Flood.pcap_Flow.csv
   DDoS-ICMP_Flood1.pcap_Flow.csv
   DDoS-ICMP_Flood10.pcap_Flow.csv
   DDoS-ICMP_Flood11.pcap_Flow.csv
   DDoS-ICMP_Flood12.pcap_Flow.csv

Folder: DDoS-HTTP Flood
CSV files: 1
Examples:
   DDoS-HTTP_Flood-.pcap_Flow.csv

Folder: DDoS-ICMP_Fragmentation
CSV files: 17
Examples:


In [12]:
import os
import pandas as pd

DIAD_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset"

OUTPUT_DIR = "/kaggle/working/O1_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

records = []

for root, dirs, files in os.walk(DIAD_PATH):
    for filename in files:
        if filename.lower().endswith(".csv"):

            folder = os.path.basename(root)

            records.append({
                "file": filename,
                "folder": folder,
                "proposed_label": folder,
                "full_path": os.path.join(root, filename)
            })

audit_df = pd.DataFrame(records)

print("PROPOSED LABEL AUDIT")
print("=" * 70)

print("\nFiles by proposed class:")
print(audit_df["proposed_label"].value_counts().sort_index())

print("\nTotal files:", len(audit_df))

print("\nMissing expected attack categories:")

expected = {
    "Benign", "Brute Force", "DDOS", "DOS",
    "Mirai", "Recon", "Spoofing", "Web-based"
}

observed = set(audit_df["folder"].unique())

for category in sorted(expected - observed):
    print("-", category)

print("\nUnexpected folders:")
for category in sorted(observed - expected):
    print("-", category)

output_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_proposed_folder_label_audit.csv"
)

audit_df.to_csv(output_path, index=False)

print("\nAudit saved to:", output_path)
print("\nIMPORTANT: These are proposed folder-derived labels, not yet verified ground truth.")

PROPOSED LABEL AUDIT

Files by proposed class:
proposed_label
ARP Spoofing                2
Benign                      4
BruteForce                  1
DDoS ACK Fragmentation     10
DDoS ICMP Flood            27
DDoS-HTTP Flood             1
DDoS-ICMP_Fragmentation    17
DNS Spoofing                1
DoS SYN Flood               8
DoS-HTTP_Flood              2
DoS-TCP_Flood               5
DoS-UDP_Flood              15
Mirai                      29
Uploading_Attack            1
VulnerabilityScan           1
XSS                         1
sqlinjection                1
Name: count, dtype: int64

Total files: 126

Missing expected attack categories:
- Brute Force
- DDOS
- DOS
- Recon
- Spoofing
- Web-based

Unexpected folders:
- ARP Spoofing
- BruteForce
- DDoS ACK Fragmentation
- DDoS ICMP Flood
- DDoS-HTTP Flood
- DDoS-ICMP_Fragmentation
- DNS Spoofing
- DoS SYN Flood
- DoS-HTTP_Flood
- DoS-TCP_Flood
- DoS-UDP_Flood
- Uploading_Attack
- VulnerabilityScan
- XSS
- sqlinjection

Audit saved 

In [13]:
import pandas as pd
import os

report_path = (
    "/kaggle/working/O1_results/"
    "O1_DIAD_corrected_file_quality.csv"
)

quality_df = pd.read_csv(report_path)

print("QUALITY REPORT COLUMNS:")
print(quality_df.columns.tolist())

print("\nFIRST 10 ROWS:")
display(quality_df.head(10))

print("\nREPORT DIMENSIONS:")
print(quality_df.shape)

QUALITY REPORT COLUMNS:
['File', 'Rows', 'Missing', 'Infinite', 'Within_Chunk_Duplicates', 'Status']

FIRST 10 ROWS:


,File,Rows,Missing,Infinite,Within_Chunk_Duplicates,Status
0,BenignTraffic.pcap_Flow.csv,183630,10,60,7,Success
1,BenignTraffic1.pcap_Flow.csv,84526,7,63,4,Success
2,BenignTraffic2.pcap_Flow.csv,91279,14,76,4,Success
3,BenignTraffic3.pcap_Flow.csv,38895,1,33,1,Success
4,DictionaryBruteForce.pcap_Flow.csv,3619,0,0,0,Success
5,DDoS-ACK_Fragmentation.pcap_Flow.csv,323886,5,15,10,Success
6,DDoS-ACK_Fragmentation1.pcap_Flow.csv,137172,2,10,5,Success
7,DDoS-ACK_Fragmentation12.pcap_Flow.csv,37292,2,10,3,Success
8,DDoS-ACK_Fragmentation2.pcap_Flow.csv,121409,12,16,6,Success
9,DDoS-ACK_Fragmentation3.pcap_Flow.csv,174443,5,5,7,Success



REPORT DIMENSIONS:
(126, 6)


In [14]:
import os
import pandas as pd

OUTPUT_DIR = "/kaggle/working/O1_results"

quality_path = os.path.join(
    OUTPUT_DIR, "O1_DIAD_corrected_file_quality.csv"
)

audit_path = os.path.join(
    OUTPUT_DIR, "O1_DIAD_proposed_folder_label_audit.csv"
)

quality_df = pd.read_csv(quality_path)
audit_df = pd.read_csv(audit_path)

# Normalize filenames for matching
def normalize_filename(value):
    return os.path.basename(str(value)).replace("\\", "").strip()

quality_df["File_key"] = quality_df["File"].apply(normalize_filename)
audit_df["File_key"] = audit_df["file"].apply(normalize_filename)

# Attach proposed folder labels to quality measurements
merged = quality_df.merge(
    audit_df[["File_key", "folder", "proposed_label"]],
    on="File_key",
    how="left",
    validate="one_to_one"
)

print("FILE MATCHING CHECK")
print("=" * 65)
print("Quality report files:", len(quality_df))
print("Matched files:", merged["proposed_label"].notna().sum())
print("Unmatched files:", merged["proposed_label"].isna().sum())

if merged["proposed_label"].isna().any():
    print("\nUnmatched filenames:")
    print(merged.loc[
        merged["proposed_label"].isna(), "File"
    ].tolist())

# Aggregate by proposed class
class_summary = (
    merged.groupby("proposed_label", dropna=False)
    .agg(
        Number_of_files=("File", "count"),
        Total_records=("Rows", "sum"),
        Missing_values=("Missing", "sum"),
        Infinite_values=("Infinite", "sum"),
        Within_chunk_duplicates=("Within_Chunk_Duplicates", "sum"),
        Successful_files=("Status", lambda x: (x == "Success").sum())
    )
    .reset_index()
)

class_summary["Missing_rate_percent"] = (
    class_summary["Missing_values"]
    / class_summary["Total_records"] * 100
).round(4)

class_summary["Infinite_rate_per_1000_records"] = (
    class_summary["Infinite_values"]
    / class_summary["Total_records"] * 1000
).round(4)

class_summary = class_summary.sort_values(
    "Total_records", ascending=False
)

print("\nPROPOSED CLASS-LEVEL DATA QUALITY SUMMARY")
print("=" * 65)
display(class_summary)

# Save report
output_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_proposed_class_quality_summary.csv"
)

class_summary.to_csv(output_path, index=False)

print("\nReport saved to:", output_path)

print("\nIMPORTANT:")
print("Class labels are provisional folder-derived labels.")
print("Duplicate counts are within-chunk counts, not global duplicates.")

FILE MATCHING CHECK
Quality report files: 126
Matched files: 126
Unmatched files: 0

PROPOSED CLASS-LEVEL DATA QUALITY SUMMARY


,proposed_label,Number_of_files,Total_records,Missing_values,Infinite_values,Within_chunk_duplicates,Successful_files,Missing_rate_percent,Infinite_rate_per_1000_records
8,DoS SYN Flood,8,16290644,1993,3449,353,8,0.0122,0.2117
11,DoS-UDP_Flood,15,5210536,3265,5003,344,15,0.0627,0.9602
3,DDoS ACK Fragmentation,10,1672474,52,118,65,10,0.0031,0.0706
9,DoS-HTTP_Flood,2,1642744,78718,80324,26677,2,4.7919,48.8962
5,DDoS-HTTP Flood,1,505720,1113,1133,4745,1,0.2201,2.2404
14,VulnerabilityScan,1,442158,521,1733,22,1,0.1178,3.9194
1,Benign,4,398330,32,232,16,4,0.0080,0.5824
6,DDoS-ICMP_Fragmentation,17,257447,92,174,254,17,0.0357,0.6759
4,DDoS ICMP Flood,27,202251,409,637,264,27,0.2022,3.1496
12,Mirai,29,174588,38,102,201,29,0.0218,0.5842



Report saved to: /kaggle/working/O1_results/O1_DIAD_proposed_class_quality_summary.csv

IMPORTANT:
Class labels are provisional folder-derived labels.
Duplicate counts are within-chunk counts, not global duplicates.


In [15]:
import os
import re
import pandas as pd

DIAD_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset"


records = []

for root, dirs, files in os.walk(DIAD_PATH):
    for filename in files:
        if filename.lower().endswith(".csv"):

            folder = os.path.basename(root)

            # Remove CSV and flow suffix
            scenario = filename.replace(".csv", "")
            scenario = scenario.replace(".pcap_Flow", "")

            # Remove trailing capture numbers
            scenario = re.sub(r"\d+$", "", scenario)

            records.append({
                "Filename": filename,
                "Folder": folder,
                "Scenario": scenario
            })

validation_df = pd.DataFrame(records)

print("FOLDER AND SCENARIO VALIDATION")
print("=" * 70)

print("Total files:", len(validation_df))

print("\nFiles per folder:")
print(validation_df["Folder"].value_counts())

print("\nUnique scenarios per folder:")

for folder, group in validation_df.groupby("Folder"):
    print(f"\n{folder}:")
    for scenario in sorted(group["Scenario"].unique()):
        print("  ", scenario)

output_path = "/kaggle/working/O1_results/O1_DIAD_scenario_validation.csv"

validation_df.to_csv(output_path, index=False)

print("\nValidation report saved:", output_path)

FOLDER AND SCENARIO VALIDATION
Total files: 126

Files per folder:
Folder
Mirai                      29
DDoS ICMP Flood            27
DDoS-ICMP_Fragmentation    17
DoS-UDP_Flood              15
DDoS ACK Fragmentation     10
DoS SYN Flood               8
DoS-TCP_Flood               5
Benign                      4
DoS-HTTP_Flood              2
ARP Spoofing                2
DDoS-HTTP Flood             1
BruteForce                  1
VulnerabilityScan           1
DNS Spoofing                1
Uploading_Attack            1
XSS                         1
sqlinjection                1
Name: count, dtype: int64

Unique scenarios per folder:

ARP Spoofing:
   MITM-ArpSpoofing

Benign:
   BenignTraffic

BruteForce:
   DictionaryBruteForce

DDoS ACK Fragmentation:
   DDoS-ACK_Fragmentation

DDoS ICMP Flood:
   DDoS-ICMP_Flood

DDoS-HTTP Flood:
   DDoS-HTTP_Flood-

DDoS-ICMP_Fragmentation:
   DDoS-ICMP_Fragmentation

DNS Spoofing:
   DNS_Spoofing

DoS SYN Flood:
   DoS-SYN_Flood

DoS-HTTP_Flood:
  

In [16]:

# ============================================================
# O1 STEP 4: DIAD DERIVED LABEL MAPPING
# CORRECTED AND EXPANDED VERSION
# ============================================================

import os
import pandas as pd

OUTPUT_DIR = "/kaggle/working/O1_results"

input_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_scenario_validation.csv"
)

if not os.path.exists(input_path):
    raise FileNotFoundError(input_path)

df = pd.read_csv(input_path)

# Normalize folder names for reliable matching
def normalize_folder(value):
    return str(value).strip().lower().replace("_", " ").replace("-", " ")

# Explicit folder-to-label mapping
label_mapping = {
    "benign": "Benign",

    # Brute force
    "bruteforce": "Brute Force",
    "brute force": "Brute Force",

    # DDoS attacks
    "ddos": "DDoS",
    "ddos ack fragmentation": "DDoS",
    "ddos icmp flood": "DDoS",
    "ddos http flood": "DDoS",
    "ddos icmp fragmentation": "DDoS",

    # DoS attacks
    "dos": "DoS",
    "dos syn flood": "DoS",
    "dos http flood": "DoS",
    "dos tcp flood": "DoS",
    "dos udp flood": "DoS",

    # Other attack categories
    "mirai": "Mirai",
    "recon": "Recon",
    "vulnerabilityscan": "Recon",
    "vulnerability scan": "Recon",

    "spoofing": "Spoofing",
    "arp spoofing": "Spoofing",
    "dns spoofing": "Spoofing",

    # Other distinct attacks
    "uploading attack": "Uploading Attack",
    "xss": "XSS",
    "sqlinjection": "SQL Injection",
    "sql injection": "SQL Injection"
}

# Normalize actual folder names
df["Normalized_Folder"] = df["Folder"].apply(
    normalize_folder
)

# Create separate derived label
df["Derived_Label"] = df["Normalized_Folder"].map(
    label_mapping
)

# Mapping report
print("=" * 65)
print("DIAD DERIVED-LABEL MAPPING VALIDATION")
print("=" * 65)

print("Total files:", len(df))

print(
    "Successfully mapped:",
    df["Derived_Label"].notna().sum()
)

print(
    "Unmapped files:",
    df["Derived_Label"].isna().sum()
)

# Identify unmapped folders
if df["Derived_Label"].isna().any():

    print("\nUNMAPPED FOLDERS:")

    print(
        df.loc[
            df["Derived_Label"].isna(),
            "Folder"
        ].value_counts()
    )

else:
    print("\nAll folders successfully mapped!")

# File counts by class
print("\nFILES PER DERIVED CLASS:")

print(
    df["Derived_Label"].value_counts(
        dropna=False
    )
)

# Scenario mapping
print("\nSCENARIO AND DERIVED LABEL:")

display(
    df[
        [
            "Filename",
            "Folder",
            "Scenario",
            "Derived_Label"
        ]
    ].head(20)
)

# Save output
output_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_file_derived_label_mapping.csv"
)

df.to_csv(output_path, index=False)

print("\nMapping report saved:")
print(output_path)

print("\nOriginal dataset labels remain unchanged.")

# Stop if any folders are unmapped
if df["Derived_Label"].isna().any():
    raise ValueError(
        "Mapping incomplete. Review unmapped folders before proceeding."
    )

print("\nO1 STEP 4 COMPLETED SUCCESSFULLY")


DIAD DERIVED-LABEL MAPPING VALIDATION
Total files: 126
Successfully mapped: 126
Unmapped files: 0

All folders successfully mapped!

FILES PER DERIVED CLASS:
Derived_Label
DDoS                55
DoS                 30
Mirai               29
Benign               4
Spoofing             3
Brute Force          1
Recon                1
Uploading Attack     1
XSS                  1
SQL Injection        1
Name: count, dtype: int64

SCENARIO AND DERIVED LABEL:


,Filename,Folder,Scenario,Derived_Label
0,BenignTraffic.pcap_Flow.csv,Benign,BenignTraffic,Benign
1,BenignTraffic1.pcap_Flow.csv,Benign,BenignTraffic,Benign
2,BenignTraffic2.pcap_Flow.csv,Benign,BenignTraffic,Benign
3,BenignTraffic3.pcap_Flow.csv,Benign,BenignTraffic,Benign
4,DictionaryBruteForce.pcap_Flow.csv,BruteForce,DictionaryBruteForce,Brute Force
5,DDoS-ACK_Fragmentation.pcap_Flow.csv,DDoS ACK Fragmentation,DDoS-ACK_Fragmentation,DDoS
6,DDoS-ACK_Fragmentation1.pcap_Flow.csv,DDoS ACK Fragmentation,DDoS-ACK_Fragmentation,DDoS
7,DDoS-ACK_Fragmentation12.pcap_Flow.csv,DDoS ACK Fragmentation,DDoS-ACK_Fragmentation,DDoS
8,DDoS-ACK_Fragmentation2.pcap_Flow.csv,DDoS ACK Fragmentation,DDoS-ACK_Fragmentation,DDoS
9,DDoS-ACK_Fragmentation3.pcap_Flow.csv,DDoS ACK Fragmentation,DDoS-ACK_Fragmentation,DDoS



Mapping report saved:
/kaggle/working/O1_results/O1_DIAD_file_derived_label_mapping.csv

Original dataset labels remain unchanged.

O1 STEP 4 COMPLETED SUCCESSFULLY


In [17]:

import os
import pandas as pd
import numpy as np

OUTPUT_DIR = "/kaggle/working/O1_results"

quality_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_corrected_file_quality.csv"
)

mapping_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_file_derived_label_mapping.csv"
)

quality = pd.read_csv(quality_path)
mapping = pd.read_csv(mapping_path)

# Match filenames
def normalize_filename(value):
    return os.path.basename(str(value)).replace("\\", "").strip()

quality["File_key"] = quality["File"].apply(normalize_filename)
mapping["File_key"] = mapping["Filename"].apply(normalize_filename)

# Merge record counts with proposed labels
verified = quality.merge(
    mapping[["File_key", "Derived_Label"]],
    on="File_key",
    how="left",
    validate="one_to_one"
)

# Check matching
print("=" * 65)
print("DERIVED-LABEL RECORD COUNT VERIFICATION")
print("=" * 65)

print("Quality report files:", len(quality))
print("Mapping report files:", len(mapping))
print("Matched files:", verified["Derived_Label"].notna().sum())
print("Unmatched files:", verified["Derived_Label"].isna().sum())

if verified["Derived_Label"].isna().any():
    raise ValueError("Unmatched files detected. Stop verification.")

# Calculate record distribution
summary = (
    verified.groupby("Derived_Label")
    .agg(
        Number_of_files=("File", "count"),
        Records=("Rows", "sum")
    )
    .reset_index()
)

total_records = summary["Records"].sum()

summary["Percentage"] = (
    summary["Records"] / total_records * 100
).round(4)

summary = summary.sort_values(
    "Records", ascending=False
)

print("\nRECORD DISTRIBUTION:")
display(summary)

print("\nTOTAL RECORDS:", total_records)

# Calculate proposed imbalance ratio
majority = summary["Records"].max()
minority = summary["Records"].min()

print("Largest class:", majority)
print("Smallest class:", minority)
print("Proposed imbalance ratio:", round(majority / minority, 2))

# Save verification report
output_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_derived_label_record_verification.csv"
)

summary.to_csv(output_path, index=False)

print("\nVerification report saved:")
print(output_path)

print("\nNOTE: These statistics use provisional folder-derived labels.")


DERIVED-LABEL RECORD COUNT VERIFICATION
Quality report files: 126
Mapping report files: 126
Matched files: 126
Unmatched files: 0

RECORD DISTRIBUTION:


,Derived_Label,Number_of_files,Records,Percentage
3,DoS,30,23143929,85.8165
2,DDoS,55,2637892,9.7812
5,Recon,1,442158,1.6395
0,Benign,4,398330,1.4770
4,Mirai,29,174588,0.6474
7,Spoofing,3,157238,0.5830
6,SQL Injection,1,6603,0.0245
1,Brute Force,1,3619,0.0134
9,XSS,1,3377,0.0125
8,Uploading Attack,1,1348,0.0050



TOTAL RECORDS: 26969082
Largest class: 23143929
Smallest class: 1348
Proposed imbalance ratio: 17169.09

Verification report saved:
/kaggle/working/O1_results/O1_DIAD_derived_label_record_verification.csv

NOTE: These statistics use provisional folder-derived labels.


In [18]:
import os
import pandas as pd

DATASET_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset"

print("Checking existing DIAD dataset...")

for root, dirs, files in os.walk(DATASET_PATH):
    csv_files = [f for f in files if f.endswith(".csv")]

    for filename in csv_files[:3]:
        path = os.path.join(root, filename)

        print("\nFile:", filename)

        df = pd.read_csv(path, nrows=5)

        print("Columns:")
        print(df.columns.tolist())

        if "Label" in df.columns:
            print("\nOriginal Label values:")
            print(df["Label"].value_counts(dropna=False))

        elif "Label 2 for AD" in df.columns:
            print("\nOriginal AD labels:")
            print(df["Label 2 for AD"].value_counts(dropna=False))

Checking existing DIAD dataset...

File: BenignTraffic.pcap_Flow.csv
Columns:
['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag C

In [19]:

import os

DATASET_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset"

print("Dataset path exists:", os.path.exists(DATASET_PATH))

print("\nFiles and folders:")
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, "").count(os.sep)
    indent = " " * 4 * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:10]:
        print(f"{indent}    {file}")

    if level >= 2:
        dirs[:] = []


Dataset path exists: True

Files and folders:
cic-iot-diad-2024-dataset/
    CIC-IoT-DIAD2024/
        README.txt.txt
        Benign/
            BenignTraffic.pcap_Flow.csv
            BenignTraffic1.pcap_Flow.csv
            BenignTraffic2.pcap_Flow.csv
            BenignTraffic3.pcap_Flow.csv
        BruteForce/
            DictionaryBruteForce.pcap_Flow.csv
        DDos/
        DoS/
        Mirai/
            Mirai-greeth_flood.pcap_Flow.csv
            Mirai-greeth_flood1.pcap_Flow.csv
            Mirai-greeth_flood10.pcap_Flow.csv
            Mirai-greeth_flood11.pcap_Flow.csv
            Mirai-greeth_flood12.pcap_Flow.csv
            Mirai-greeth_flood13.pcap_Flow.csv
            Mirai-greeth_flood14.pcap_Flow.csv
            Mirai-greeth_flood15.pcap_Flow.csv
            Mirai-greeth_flood16.pcap_Flow.csv
            Mirai-greeth_flood17.pcap_Flow.csv
        Recon/
        Spoofing/
        Web-Based/


In [20]:

import os
import pandas as pd

DATASET_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024"

# 1. Read the dataset README
readme_path = os.path.join(DATASET_PATH, "README.txt.txt")

print("=" * 70)
print("1. README CONTENT")
print("=" * 70)

if os.path.exists(readme_path):
    with open(readme_path, "r", encoding="utf-8", errors="replace") as f:
        print(f.read()[:12000])
else:
    print("README file not found.")

# 2. Count CSV files in each category
print("\n" + "=" * 70)
print("2. CSV FILE COUNTS BY CATEGORY")
print("=" * 70)

category_files = {}

for category in sorted(os.listdir(DATASET_PATH)):
    folder = os.path.join(DATASET_PATH, category)

    if os.path.isdir(folder):
        csv_files = []

        for root, dirs, files in os.walk(folder):
            for file in files:
                if file.lower().endswith(".csv"):
                    csv_files.append(os.path.join(root, file))

        category_files[category] = sorted(csv_files)
        print(f"{category}: {len(csv_files)} CSV files")

# 3. Inspect one CSV from each category
print("\n" + "=" * 70)
print("3. SAMPLE CSV SCHEMA AND LABEL CHECK")
print("=" * 70)

for category, files in category_files.items():
    print(f"\n--- CATEGORY: {category} ---")

    if not files:
        print("No CSV files found.")
        continue

    sample_file = files[0]

    try:
        df = pd.read_csv(sample_file, nrows=20, low_memory=False)

        print("Sample file:", os.path.basename(sample_file))
        print("Number of columns:", len(df.columns))
        print("First 10 columns:", df.columns.tolist()[:10])

        label_columns = [
            col for col in df.columns
            if col.strip().lower() == "label"
        ]

        if label_columns:
            label_col = label_columns[0]
            print("Label column:", label_col)
            print("Sample label values:")
            print(df[label_col].value_counts(dropna=False).to_string())
        else:
            print("WARNING: No Label column found.")

    except Exception as e:
        print("Could not inspect file:", str(e))


1. README CONTENT
The CICFlowMeter is an open source tool that generates Biflows from pcap files, and extracts features from these flows.



NOTE: TCP flows are usually terminated upon connection teardown (by FIN packet) while UDP flows are terminated by a flow timeout. The flow timeout value can be assigned arbitrarily by the individual scheme e.g., 600 seconds for both TCP and UDP.


NOTE:

The data found in this folder is comprised of 8 folders(Benign + 7 Attacks). Each folder will contain csv files for the kind of traffic specified by the folder name.

The data has not been labeled on a row basis.

But as per the individual, they can labeled it as Benign and Attack(or Attack Name). The naming of attack can be decided on the level of granularity that the experiment requires.





--------------------------------------------------------------
List of extracted features and descriptions:
Feature Name				Description
Flow duration			Duration of the flow in Microsecond
total Fwd Packet		

In [21]:

import os
import pandas as pd

DATASET_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024"

rows = []

for category in sorted(os.listdir(DATASET_PATH)):
    folder = os.path.join(DATASET_PATH, category)

    if not os.path.isdir(folder):
        continue

    for root, dirs, files in os.walk(folder):
        for filename in sorted(files):
            if filename.lower().endswith(".csv"):
                rows.append({
                    "Proposed_Class": category,
                    "Scenario_File": filename,
                    "File_Path": os.path.join(root, filename)
                })

manifest = pd.DataFrame(rows)

print("Total CSV files:", len(manifest))
print("\nFiles per proposed class:")
print(manifest.groupby("Proposed_Class").size().to_string())

print("\nComplete file-to-class mapping:")
print(
    manifest[["Proposed_Class", "Scenario_File"]]
    .sort_values(["Proposed_Class", "Scenario_File"])
    .to_string(index=False)
)

output_path = "/kaggle/working/O1_DIAD_new_file_manifest.csv"
manifest.to_csv(output_path, index=False)

print("\nManifest saved to:", output_path)


Total CSV files: 126

Files per proposed class:
Proposed_Class
Benign         4
BruteForce     1
DDos          55
DoS           30
Mirai         29
Recon          1
Spoofing       3
Web-Based      3

Complete file-to-class mapping:
Proposed_Class                           Scenario_File
        Benign             BenignTraffic.pcap_Flow.csv
        Benign            BenignTraffic1.pcap_Flow.csv
        Benign            BenignTraffic2.pcap_Flow.csv
        Benign            BenignTraffic3.pcap_Flow.csv
    BruteForce      DictionaryBruteForce.pcap_Flow.csv
          DDos    DDoS-ACK_Fragmentation.pcap_Flow.csv
          DDos   DDoS-ACK_Fragmentation1.pcap_Flow.csv
          DDos  DDoS-ACK_Fragmentation12.pcap_Flow.csv
          DDos   DDoS-ACK_Fragmentation2.pcap_Flow.csv
          DDos   DDoS-ACK_Fragmentation3.pcap_Flow.csv
          DDos   DDoS-ACK_Fragmentation4.pcap_Flow.csv
          DDos   DDoS-ACK_Fragmentation5.pcap_Flow.csv
          DDos   DDoS-ACK_Fragmentation6.pcap_Flow.cs

In [22]:

import os
import pandas as pd

DATASET_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024"

# Map folder names to standardized class names
CLASS_MAP = {
    "Benign": "Benign",
    "BruteForce": "BruteForce",
    "DDos": "DDoS",
    "DoS": "DoS",
    "Mirai": "Mirai",
    "Recon": "Recon",
    "Spoofing": "Spoofing",
    "Web-Based": "Web-Based"
}

print("=" * 75)
print("ROW-LEVEL LABEL ASSIGNMENT VALIDATION")
print("=" * 75)

results = []

for folder_name, proposed_class in CLASS_MAP.items():
    folder_path = os.path.join(DATASET_PATH, folder_name)

    csv_files = sorted([
        f for f in os.listdir(folder_path)
        if f.lower().endswith(".csv")
    ])

    if not csv_files:
        print(f"WARNING: No CSV files in {folder_name}")
        continue

    sample_path = os.path.join(folder_path, csv_files[0])

    df = pd.read_csv(sample_path, nrows=100, low_memory=False)

    results.append({
        "Folder": folder_name,
        "Sample_File": csv_files[0],
        "Proposed_Class": proposed_class,
        "Rows_Checked": len(df),
        "Original_Label_Values": (
            df["Label"].astype(str).unique().tolist()
            if "Label" in df.columns else "Missing"
        ),
        "Assigned_Label": proposed_class
    })

validation = pd.DataFrame(results)

print(validation.to_string(index=False))

print("\nCategories checked:", len(validation))
print("Total sample rows checked:", validation["Rows_Checked"].sum())
print("All categories represented:", len(validation) == 8)
print("Original Label retained; proposed labels stored separately.")

output_path = "/kaggle/working/O1_DIAD_label_assignment_validation.csv"
validation.to_csv(output_path, index=False)

print("\nValidation report saved:", output_path)


ROW-LEVEL LABEL ASSIGNMENT VALIDATION
    Folder                        Sample_File Proposed_Class  Rows_Checked Original_Label_Values Assigned_Label
    Benign        BenignTraffic.pcap_Flow.csv         Benign           100     [NeedManualLabel]         Benign
BruteForce DictionaryBruteForce.pcap_Flow.csv     BruteForce           100     [NeedManualLabel]     BruteForce
     Mirai   Mirai-greeth_flood.pcap_Flow.csv          Mirai           100     [NeedManualLabel]          Mirai

Categories checked: 3
Total sample rows checked: 300
All categories represented: False
Original Label retained; proposed labels stored separately.

Validation report saved: /kaggle/working/O1_DIAD_label_assignment_validation.csv


In [23]:

import os
import pandas as pd

DATASET_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024"

CLASS_MAP = {
    "Benign": "Benign",
    "BruteForce": "BruteForce",
    "DDos": "DDoS",
    "DoS": "DoS",
    "Mirai": "Mirai",
    "Recon": "Recon",
    "Spoofing": "Spoofing",
    "Web-Based": "Web-Based"
}

results = []

print("=" * 75)
print("CORRECTED ROW-LEVEL LABEL ASSIGNMENT VALIDATION")
print("=" * 75)

for folder_name, proposed_class in CLASS_MAP.items():

    folder_path = os.path.join(DATASET_PATH, folder_name)

    # Search recursively inside the category folder
    csv_paths = []

    for root, dirs, files in os.walk(folder_path):
        for filename in files:
            if filename.lower().endswith(".csv"):
                csv_paths.append(os.path.join(root, filename))

    csv_paths = sorted(csv_paths)

    if not csv_paths:
        print(f"WARNING: No CSV files found in {folder_name}")
        continue

    sample_path = csv_paths[0]

    try:
        df = pd.read_csv(sample_path, nrows=100, low_memory=False)

        original_labels = (
            df["Label"].astype(str).unique().tolist()
            if "Label" in df.columns
            else ["Label column missing"]
        )

        results.append({
            "Folder": folder_name,
            "CSV_Count": len(csv_paths),
            "Sample_File": os.path.basename(sample_path),
            "Proposed_Class": proposed_class,
            "Rows_Checked": len(df),
            "Original_Label_Values": str(original_labels),
            "Assigned_Label": proposed_class
        })

    except Exception as e:
        print(f"ERROR reading {sample_path}: {e}")

validation = pd.DataFrame(results)

print("\nVALIDATION RESULTS:")
print(validation.to_string(index=False))

print("\nCategories successfully checked:", len(validation))
print("Total sample rows checked:", validation["Rows_Checked"].sum())
print("All eight categories found:", len(validation) == 8)

output_path = "/kaggle/working/O1_DIAD_label_assignment_validation.csv"
validation.to_csv(output_path, index=False)

print("\nReport saved:", output_path)


CORRECTED ROW-LEVEL LABEL ASSIGNMENT VALIDATION

VALIDATION RESULTS:
    Folder  CSV_Count                          Sample_File Proposed_Class  Rows_Checked Original_Label_Values Assigned_Label
    Benign          4          BenignTraffic.pcap_Flow.csv         Benign           100   ['NeedManualLabel']         Benign
BruteForce          1   DictionaryBruteForce.pcap_Flow.csv     BruteForce           100   ['NeedManualLabel']     BruteForce
      DDos         55 DDoS-ACK_Fragmentation.pcap_Flow.csv           DDoS           100   ['NeedManualLabel']           DDoS
       DoS         30          DoS-SYN_Flood.pcap_Flow.csv            DoS           100   ['NeedManualLabel']            DoS
     Mirai         29     Mirai-greeth_flood.pcap_Flow.csv          Mirai           100   ['NeedManualLabel']          Mirai
     Recon          1      VulnerabilityScan.pcap_Flow.csv          Recon           100   ['NeedManualLabel']          Recon
  Spoofing          3       MITM-ArpSpoofing.pcap_Flow.c

In [24]:

import os
import pandas as pd

DATASET_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024"
OUTPUT_DIR = "/kaggle/working/O1_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_MAP = {
    "Benign": "Benign",
    "BruteForce": "BruteForce",
    "DDos": "DDoS",
    "DoS": "DoS",
    "Mirai": "Mirai",
    "Recon": "Recon",
    "Spoofing": "Spoofing",
    "Web-Based": "Web-Based"
}

file_results = []

print("Counting records in all DIAD CSV files...")

for folder_name, proposed_class in CLASS_MAP.items():

    folder_path = os.path.join(DATASET_PATH, folder_name)

    for root, dirs, files in os.walk(folder_path):
        for filename in sorted(files):

            if not filename.lower().endswith(".csv"):
                continue

            file_path = os.path.join(root, filename)

            try:
                row_count = 0

                # Count records in chunks to limit memory use
                for chunk in pd.read_csv(
                    file_path,
                    usecols=["Label"],
                    chunksize=200000,
                    low_memory=False
                ):
                    row_count += len(chunk)

                file_results.append({
                    "Folder": folder_name,
                    "Proposed_Class": proposed_class,
                    "File": filename,
                    "Records": row_count,
                    "Status": "Success"
                })

                print(f"{filename}: {row_count:,} records")

            except Exception as e:
                file_results.append({
                    "Folder": folder_name,
                    "Proposed_Class": proposed_class,
                    "File": filename,
                    "Records": 0,
                    "Status": str(e)
                })

                print(f"ERROR: {filename}: {e}")

# File-level report
file_report = pd.DataFrame(file_results)

file_report.to_csv(
    os.path.join(OUTPUT_DIR, "O1_DIAD_new_file_record_counts.csv"),
    index=False
)

# Class-level distribution
class_report = (
    file_report[file_report["Status"] == "Success"]
    .groupby("Proposed_Class", as_index=False)
    .agg(
        Files=("File", "count"),
        Records=("Records", "sum")
    )
)

total_records = class_report["Records"].sum()

class_report["Percentage"] = (
    class_report["Records"] / total_records * 100
)

class_report = class_report.sort_values(
    "Records", ascending=False
)

print("\n" + "=" * 70)
print("COMPLETE DIAD CLASS DISTRIBUTION")
print("=" * 70)

print(class_report.to_string(index=False))

print("\nTotal CSV files processed:", len(file_report))
print("Successful files:", (file_report["Status"] == "Success").sum())
print("Failed files:", (file_report["Status"] != "Success").sum())
print("Total records:", f"{total_records:,}")

if len(class_report) > 1 and class_report["Records"].min() > 0:
    imbalance_ratio = (
        class_report["Records"].max() /
        class_report["Records"].min()
    )
    print("Largest-to-smallest class ratio:", round(imbalance_ratio, 2), ":1")

class_report.to_csv(
    os.path.join(OUTPUT_DIR, "O1_DIAD_new_class_distribution.csv"),
    index=False
)

print("\nReports saved in:", OUTPUT_DIR)


Counting records in all DIAD CSV files...
BenignTraffic.pcap_Flow.csv: 183,630 records
BenignTraffic1.pcap_Flow.csv: 84,526 records
BenignTraffic2.pcap_Flow.csv: 91,279 records
BenignTraffic3.pcap_Flow.csv: 38,895 records
DictionaryBruteForce.pcap_Flow.csv: 3,619 records
DDoS-ACK_Fragmentation.pcap_Flow.csv: 323,886 records
DDoS-ACK_Fragmentation1.pcap_Flow.csv: 137,172 records
DDoS-ACK_Fragmentation12.pcap_Flow.csv: 37,292 records
DDoS-ACK_Fragmentation2.pcap_Flow.csv: 121,409 records
DDoS-ACK_Fragmentation3.pcap_Flow.csv: 174,443 records
DDoS-ACK_Fragmentation4.pcap_Flow.csv: 152,216 records
DDoS-ACK_Fragmentation5.pcap_Flow.csv: 209,344 records
DDoS-ACK_Fragmentation6.pcap_Flow.csv: 209,790 records
DDoS-ACK_Fragmentation7.pcap_Flow.csv: 127,543 records
DDoS-ACK_Fragmentation8.pcap_Flow.csv: 179,379 records
DDoS-ICMP_Flood.pcap_Flow.csv: 10,415 records
DDoS-ICMP_Flood1.pcap_Flow.csv: 8,263 records
DDoS-ICMP_Flood10.pcap_Flow.csv: 7,558 records
DDoS-ICMP_Flood11.pcap_Flow.csv: 8,312 r

In [25]:

import os
import pandas as pd

DATASET_PATH = "/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024"

target_files = {
    "DoS-TCP_Flood.pcap_Flow.csv",
    "DoS-TCP_Flood1.pcap_Flow.csv",
    "DoS-TCP_Flood10.pcap_Flow.csv",
    "DoS-TCP_Flood2.pcap_Flow.csv",
    "DoS-TCP_Flood3.pcap_Flow.csv"
}

# Obtain the reference schema from a normal headered CSV
reference_path = os.path.join(
    DATASET_PATH, "Benign", "BenignTraffic.pcap_Flow.csv"
)

reference_columns = pd.read_csv(reference_path, nrows=0).columns.tolist()

print("Reference column count:", len(reference_columns))

results = []

for root, dirs, files in os.walk(DATASET_PATH):
    for filename in sorted(files):

        if filename not in target_files:
            continue

        file_path = os.path.join(root, filename)

        try:
            # Check the number of fields in the first data row
            sample = pd.read_csv(
                file_path,
                header=None,
                nrows=2,
                low_memory=False
            )

            field_count = sample.shape[1]

            if field_count != len(reference_columns):
                print(
                    f"SCHEMA WARNING: {filename}: "
                    f"{field_count} fields; expected {len(reference_columns)}"
                )
                continue

            # Count rows without treating the first data row as a header
            total_rows = 0

            for chunk in pd.read_csv(
                file_path,
                header=None,
                chunksize=200000,
                low_memory=False
            ):
                total_rows += len(chunk)

            results.append({
                "File": filename,
                "Columns": field_count,
                "Records": total_rows,
                "Status": "Success"
            })

            print(f"{filename}: {total_rows:,} records")

        except Exception as e:
            print(f"ERROR: {filename}: {e}")

headerless_report = pd.DataFrame(results)

print("\n" + "=" * 65)
print("HEADERLESS FILE COUNTING RESULTS")
print("=" * 65)

if not headerless_report.empty:
    print(headerless_report.to_string(index=False))
    print("\nSuccessfully counted:", len(headerless_report))
    print(
        "Additional DoS records:",
        f"{headerless_report['Records'].sum():,}"
    )

    output_path = "/kaggle/working/O1_results/O1_DIAD_headerless_file_counts.csv"
    headerless_report.to_csv(output_path, index=False)
    print("\nReport saved:", output_path)
else:
    print("No files were successfully counted.")


Reference column count: 84
DoS-TCP_Flood.pcap_Flow.csv: 1 records
DoS-TCP_Flood1.pcap_Flow.csv: 1 records
DoS-TCP_Flood10.pcap_Flow.csv: 1 records
DoS-TCP_Flood2.pcap_Flow.csv: 1 records
DoS-TCP_Flood3.pcap_Flow.csv: 1 records

HEADERLESS FILE COUNTING RESULTS
                         File  Columns  Records  Status
  DoS-TCP_Flood.pcap_Flow.csv       84        1 Success
 DoS-TCP_Flood1.pcap_Flow.csv       84        1 Success
DoS-TCP_Flood10.pcap_Flow.csv       84        1 Success
 DoS-TCP_Flood2.pcap_Flow.csv       84        1 Success
 DoS-TCP_Flood3.pcap_Flow.csv       84        1 Success

Successfully counted: 5
Additional DoS records: 5

Report saved: /kaggle/working/O1_results/O1_DIAD_headerless_file_counts.csv


In [26]:

import os
import pandas as pd

# ============================================================
# HEADERLESS DOS-TCP FILE INSPECTION - CORRECTED VERSION
# ============================================================

DATASET_PATH = (
    "/kaggle/input/datasets/"
    "mostafaaliogab/cic-iot-diad-2024-dataset"
)

headerless_files = [
    "DoS-TCP_Flood.pcap_Flow.csv",
    "DoS-TCP_Flood1.pcap_Flow.csv",
    "DoS-TCP_Flood10.pcap_Flow.csv",
    "DoS-TCP_Flood2.pcap_Flow.csv",
    "DoS-TCP_Flood3.pcap_Flow.csv"
]

print("=" * 80)
print("SEARCHING FOR HEADERLESS DOS-TCP FILES")
print("=" * 80)

# Check dataset directory
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Dataset directory does not exist: {DATASET_PATH}\n"
        "Check the Kaggle dataset path in the Input panel."
    )

# Search recursively for all CSV files
found_files = {}

for root, dirs, files in os.walk(DATASET_PATH):
    for filename in files:
        if filename.lower().endswith(".csv"):
            normalized = filename.replace("\\", "/").strip()
            if normalized in headerless_files:
                found_files[normalized] = os.path.join(root, filename)

# ============================================================
# INSPECT FILES
# ============================================================

results = []

for filename in headerless_files:

    print("\n" + "-" * 80)
    print("FILE:", filename)

    path = found_files.get(filename)

    if path is None:
        print("STATUS: FILE NOT FOUND")
        results.append({
            "File": filename,
            "File_Size_Bytes": None,
            "Physical_Lines": None,
            "Comma_Count": None,
            "Pandas_Rows": None,
            "Pandas_Columns": None,
            "Status": "Not found"
        })
        continue

    print("Actual path:", path)

    with open(path, "r", encoding="utf-8-sig",
              errors="replace") as f:
        content = f.read()

    file_size = os.path.getsize(path)
    physical_lines = len(content.splitlines())
    comma_count = content.count(",")

    try:
        df_test = pd.read_csv(
            path,
            header=None,
            low_memory=False
        )

        parsed_rows, parsed_columns = df_test.shape
        status = "Success"

    except Exception as e:
        parsed_rows = None
        parsed_columns = None
        status = str(e)

    print("File size:", f"{file_size:,}", "bytes")
    print("Physical lines:", f"{physical_lines:,}")
    print("Comma count:", f"{comma_count:,}")
    print("Pandas rows:", parsed_rows)
    print("Pandas columns:", parsed_columns)
    print("First 250 characters:")
    print(repr(content[:250]))

    results.append({
        "File": filename,
        "File_Size_Bytes": file_size,
        "Physical_Lines": physical_lines,
        "Comma_Count": comma_count,
        "Pandas_Rows": parsed_rows,
        "Pandas_Columns": parsed_columns,
        "Status": status
    })

# ============================================================
# SAVE INSPECTION REPORT
# ============================================================

results_df = pd.DataFrame(results)

output_folder = "/kaggle/working/O1_results"
os.makedirs(output_folder, exist_ok=True)

output_file = os.path.join(
    output_folder,
    "O1_DIAD_headerless_file_inspection.csv"
)

results_df.to_csv(output_file, index=False)

print("\n" + "=" * 80)
print("FINAL INSPECTION SUMMARY")
print("=" * 80)

print(results_df.to_string(index=False))

print("\nReport saved:", output_file)


SEARCHING FOR HEADERLESS DOS-TCP FILES

--------------------------------------------------------------------------------
FILE: DoS-TCP_Flood.pcap_Flow.csv
Actual path: /kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024/DoS/DoS-TCP_Flood/DoS-TCP_Flood.pcap_Flow.csv
File size: 460 bytes
Physical lines: 1
Comma count: 83
Pandas rows: 1
Pandas columns: 84
First 250 characters:
'35.185.101.66-192.168.137.72-443-33777-6,35.185.101.66,443,192.168.137.72,33777,6,22/07/2022 11:41:31 AM,9592154,51,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,5.316845413449367,191843.07999999996,219603.38466303944,1074112.0,33354.0,9592154.0,191843.07999'

--------------------------------------------------------------------------------
FILE: DoS-TCP_Flood1.pcap_Flow.csv
Actual path: /kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024/DoS/DoS-TCP_Flood/DoS-TCP_Flood1.pcap_Flow.csv
File size: 558 bytes
Physical lines: 1
Comma count: 83
Pandas rows: 1
Pandas column

In [27]:

import os
import pandas as pd

# ============================================================
# O1: COMPLETE CIC IoT-DIAD 2024 RECORD COUNTING
# ============================================================

DATASET_PATH = (
    "/kaggle/input/datasets/"
    "mostafaaliogab/cic-iot-diad-2024-dataset"
)

OUTPUT_FOLDER = "/kaggle/working/O1_results"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

HEADERLESS_FILES = {
    "DoS-TCP_Flood.pcap_Flow.csv",
    "DoS-TCP_Flood1.pcap_Flow.csv",
    "DoS-TCP_Flood10.pcap_Flow.csv",
    "DoS-TCP_Flood2.pcap_Flow.csv",
    "DoS-TCP_Flood3.pcap_Flow.csv"
}

CHUNK_SIZE = 200_000

# ============================================================
# 1. FIND ALL CSV FILES
# ============================================================

all_csv_files = []

for root, dirs, files in os.walk(DATASET_PATH):
    for filename in files:
        if filename.lower().endswith(".csv"):
            all_csv_files.append(
                os.path.join(root, filename)
            )

all_csv_files.sort()

print("=" * 75)
print("CIC IoT-DIAD 2024 COMPLETE RECORD COUNTING")
print("=" * 75)

print("Total CSV files found:", len(all_csv_files))

if len(all_csv_files) != 126:
    print("WARNING: Expected 126 files. Verify dataset completeness.")

# ============================================================
# 2. COUNT RECORDS FILE BY FILE
# ============================================================

file_results = []

for index, path in enumerate(all_csv_files, start=1):

    filename = os.path.basename(path)

    relative_path = os.path.relpath(
        path, DATASET_PATH
    )

    path_parts = relative_path.split(os.sep)

    category = (
        path_parts[-3] if len(path_parts) >= 3
        else "Unknown"
    )

    scenario = (
        path_parts[-2] if len(path_parts) >= 2
        else "Unknown"
    )

    is_headerless = filename in HEADERLESS_FILES

    header = None if is_headerless else 0

    try:
        total_records = 0

        reader = pd.read_csv(
            path,
            header=header,
            usecols=[0],
            dtype=str,
            chunksize=CHUNK_SIZE,
            encoding="utf-8-sig",
            on_bad_lines="error"
        )

        for chunk in reader:
            total_records += len(chunk)

        status = "Success"

    except Exception as e:
        total_records = None
        status = str(e)

    file_results.append({
        "File": filename,
        "Category": category,
        "Scenario": scenario,
        "Headerless": is_headerless,
        "Records": total_records,
        "Status": status
    })

    print(
        f"[{index}/{len(all_csv_files)}] "
        f"{filename}: {total_records} records"
        if total_records is not None
        else f"[{index}/{len(all_csv_files)}] "
             f"{filename}: ERROR - {status}"
    )

# ============================================================
# 3. CREATE FILE-LEVEL REPORT
# ============================================================

file_df = pd.DataFrame(file_results)

file_report = os.path.join(
    OUTPUT_FOLDER,
    "O1_DIAD_new_file_record_counts.csv"
)

file_df.to_csv(file_report, index=False)

# ============================================================
# 4. CATEGORY-LEVEL RECORD DISTRIBUTION
# ============================================================

successful_df = file_df[
    file_df["Status"] == "Success"
].copy()

category_distribution = (
    successful_df.groupby("Category", as_index=False)
    .agg(
        Total_Files=("File", "count"),
        Total_Records=("Records", "sum")
    )
)

category_distribution["Percentage"] = (
    category_distribution["Total_Records"]
    / category_distribution["Total_Records"].sum()
    * 100
)

category_distribution = category_distribution.sort_values(
    "Total_Records", ascending=False
)

category_report = os.path.join(
    OUTPUT_FOLDER,
    "O1_DIAD_new_class_distribution.csv"
)

category_distribution.to_csv(
    category_report, index=False
)

# ============================================================
# 5. SCENARIO-LEVEL RECORD DISTRIBUTION
# ============================================================

scenario_distribution = (
    successful_df.groupby(
        ["Category", "Scenario"], as_index=False
    )
    .agg(
        Total_Files=("File", "count"),
        Total_Records=("Records", "sum")
    )
)

scenario_distribution["Percentage"] = (
    scenario_distribution["Total_Records"]
    / scenario_distribution["Total_Records"].sum()
    * 100
)

scenario_distribution = scenario_distribution.sort_values(
    "Total_Records", ascending=False
)

scenario_report = os.path.join(
    OUTPUT_FOLDER,
    "O1_DIAD_scenario_record_distribution.csv"
)

scenario_distribution.to_csv(
    scenario_report, index=False
)

# ============================================================
# 6. FINAL VALIDATION SUMMARY
# ============================================================

total_files = len(file_df)
successful_files = len(successful_df)
failed_files = total_files - successful_files

total_records = successful_df["Records"].sum()

print("\n" + "=" * 75)
print("FINAL DIAD RECORD COUNTING SUMMARY")
print("=" * 75)

print("Total CSV files:", total_files)
print("Successfully counted:", successful_files)
print("Failed files:", failed_files)
print("Verified headerless files:",
      int(file_df["Headerless"].sum()))

print("Total counted traffic records:",
      f"{total_records:,}")

print("\nCATEGORY-LEVEL DISTRIBUTION")
print(category_distribution.to_string(index=False))

print("\nSCENARIO-LEVEL DISTRIBUTION")
print(scenario_distribution.to_string(index=False))

print("\nREPORTS SAVED:")
print(file_report)
print(category_report)
print(scenario_report)

print("\n" + "=" * 75)
print("RECORD COUNTING COMPLETED")
print("=" * 75)


CIC IoT-DIAD 2024 COMPLETE RECORD COUNTING
Total CSV files found: 126
[1/126] BenignTraffic.pcap_Flow.csv: 183630 records
[2/126] BenignTraffic1.pcap_Flow.csv: 84526 records
[3/126] BenignTraffic2.pcap_Flow.csv: 91279 records
[4/126] BenignTraffic3.pcap_Flow.csv: 38895 records
[5/126] DictionaryBruteForce.pcap_Flow.csv: 3619 records
[6/126] DDoS-ACK_Fragmentation.pcap_Flow.csv: 323886 records
[7/126] DDoS-ACK_Fragmentation1.pcap_Flow.csv: 137172 records
[8/126] DDoS-ACK_Fragmentation12.pcap_Flow.csv: 37292 records
[9/126] DDoS-ACK_Fragmentation2.pcap_Flow.csv: 121409 records
[10/126] DDoS-ACK_Fragmentation3.pcap_Flow.csv: 174443 records
[11/126] DDoS-ACK_Fragmentation4.pcap_Flow.csv: 152216 records
[12/126] DDoS-ACK_Fragmentation5.pcap_Flow.csv: 209344 records
[13/126] DDoS-ACK_Fragmentation6.pcap_Flow.csv: 209790 records
[14/126] DDoS-ACK_Fragmentation7.pcap_Flow.csv: 127543 records
[15/126] DDoS-ACK_Fragmentation8.pcap_Flow.csv: 179379 records
[16/126] DDoS-ICMP_Flood.pcap_Flow.csv: 

In [28]:

import os
import pandas as pd

# ============================================================
# CORRECT DIAD CATEGORY DISTRIBUTION
# ============================================================

OUTPUT_FOLDER = "/kaggle/working/O1_results"

input_file = os.path.join(
    OUTPUT_FOLDER,
    "O1_DIAD_new_file_record_counts.csv"
)

df = pd.read_csv(input_file)

# Verify required columns
required = ["Category", "Scenario", "Records", "Status"]

for col in required:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# Preserve original grouping for auditing
df["Original_Category"] = df["Category"]

# Correct the 34 files grouped under the dataset root
root_mask = (
    df["Category"].astype(str).str.strip()
    == "CIC-IoT-DIAD2024"
)

print("Files requiring category correction:",
      int(root_mask.sum()))

# Their scenario folders identify their actual categories
df.loc[root_mask, "Category"] = (
    df.loc[root_mask, "Scenario"]
)

# Standardize category names
category_mapping = {
    "DDos": "DDoS",
    "BruteForce": "Brute Force"
}

df["Category"] = df["Category"].replace(
    category_mapping
)

# Use successfully counted files only
valid = df[df["Status"] == "Success"].copy()

# Category distribution
class_distribution = (
    valid.groupby("Category", as_index=False)
    .agg(
        Total_Files=("Records", "count"),
        Total_Records=("Records", "sum")
    )
)

grand_total = class_distribution[
    "Total_Records"
].sum()

class_distribution["Percentage"] = (
    class_distribution["Total_Records"]
    / grand_total * 100
)

class_distribution = class_distribution.sort_values(
    "Total_Records",
    ascending=False
)

# Validation
print("\n" + "=" * 70)
print("CORRECTED DIAD CLASS DISTRIBUTION")
print("=" * 70)

print(class_distribution.to_string(index=False))

print("\nTotal records:", f"{grand_total:,}")
print("Total files:", len(valid))
print("Failed files:", len(df) - len(valid))

assert len(valid) == 126
assert grand_total == 26969082
assert root_mask.sum() == 34

# Save corrected file-level report
corrected_file = os.path.join(
    OUTPUT_FOLDER,
    "O1_DIAD_corrected_file_record_counts.csv"
)

df.to_csv(corrected_file, index=False)

# Save corrected class distribution
corrected_class = os.path.join(
    OUTPUT_FOLDER,
    "O1_DIAD_corrected_class_distribution.csv"
)

class_distribution.to_csv(
    corrected_class, index=False
)

print("\nCorrected reports saved:")
print(corrected_file)
print(corrected_class)

print("\nCATEGORY CORRECTION COMPLETED")


Files requiring category correction: 34

CORRECTED DIAD CLASS DISTRIBUTION
   Category  Total_Files  Total_Records  Percentage
        DoS           30       23143929   85.816525
       DDoS           55        2637892    9.781171
      Recon            1         442158    1.639500
     Benign            4         398330    1.476988
      Mirai           29         174588    0.647364
   Spoofing            3         157238    0.583031
  Web-Based            3          11328    0.042004
Brute Force            1           3619    0.013419

Total records: 26,969,082
Total files: 126
Failed files: 0

Corrected reports saved:
/kaggle/working/O1_results/O1_DIAD_corrected_file_record_counts.csv
/kaggle/working/O1_results/O1_DIAD_corrected_class_distribution.csv

CATEGORY CORRECTION COMPLETED


In [29]:

# ============================================================
# OBJECTIVE 1: MISSING-VALUE ANALYSIS
# Dataset: CIC IoT-DIAD 2024
# Processes files in chunks to reduce memory usage
# ============================================================

import os
import glob
import pandas as pd
import numpy as np

# Dataset location
DATASET_PATH = (
    "/kaggle/input/datasets/mostafaaliogab/"
    "cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024"
)

OUTPUT_DIR = "/kaggle/working/O1_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 100000

# Find CSV files
csv_files = glob.glob(
    os.path.join(DATASET_PATH, "**", "*.csv"),
    recursive=True
)

print("Total CSV files:", len(csv_files))

# Use the known 84-column DIAD schema from the earlier audit.
# Headerless files use the same schema as the reference file.
reference_file = next(
    (f for f in csv_files
     if os.path.basename(f) == "BenignTraffic.pcap_Flow.csv"),
    None
)

if reference_file is None:
    raise FileNotFoundError(
        "Reference file BenignTraffic.pcap_Flow.csv not found. "
        "Check DATASET_PATH."
    )

reference_columns = pd.read_csv(
    reference_file, nrows=0
).columns.tolist()

print("Reference columns:", len(reference_columns))

# Detect the five known headerless files
headerless_files = {
    f for f in csv_files
    if (
        "DoS-TCP_Flood" in f
        and os.path.basename(f).startswith("DoS-TCP_Flood")
    )
}

missing_counts = pd.Series(0, index=reference_columns, dtype="int64")
total_records = 0
failed_files = []

for i, file_path in enumerate(csv_files, 1):

    try:
        if file_path in headerless_files:
            reader = pd.read_csv(
                file_path,
                header=None,
                names=reference_columns,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )
        else:
            reader = pd.read_csv(
                file_path,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )

        file_records = 0

        for chunk in reader:

            # Count missing values in each column
            chunk_missing = chunk.isna().sum()

            for col in reference_columns:
                if col in chunk_missing.index:
                    missing_counts[col] += int(chunk_missing[col])

            file_records += len(chunk)

        total_records += file_records

        print(
            f"[{i}/{len(csv_files)}] "
            f"{os.path.basename(file_path)}: "
            f"{file_records:,} records"
        )

    except Exception as e:
        failed_files.append({
            "file": file_path,
            "error": str(e)
        })
        print("ERROR:", file_path, str(e))

# Create missing-value report
missing_report = pd.DataFrame({
    "Feature": missing_counts.index,
    "Missing_Count": missing_counts.values
})

missing_report["Missing_Percentage"] = (
    missing_report["Missing_Count"] / total_records * 100
    if total_records else 0
)

missing_report = missing_report.sort_values(
    "Missing_Percentage",
    ascending=False
)

missing_report.to_csv(
    os.path.join(OUTPUT_DIR, "O1_DIAD_missing_values.csv"),
    index=False
)

# Summary
print("\n" + "=" * 65)
print("MISSING-VALUE ANALYSIS SUMMARY")
print("=" * 65)

print("Files found:", len(csv_files))
print("Files processed:", len(csv_files) - len(failed_files))
print("Failed files:", len(failed_files))
print("Total records processed:", f"{total_records:,}")
print("Total missing values:", f"{missing_counts.sum():,}")
print(
    "Features with missing values:",
    int((missing_counts > 0).sum())
)
print(
    "Features without missing values:",
    int((missing_counts == 0).sum())
)

print("\nTOP 20 FEATURES BY MISSING PERCENTAGE:")
display(missing_report.head(20))

if failed_files:
    print("\nFAILED FILES:")
    display(pd.DataFrame(failed_files))

print("\nReport saved to:")
print(os.path.join(OUTPUT_DIR, "O1_DIAD_missing_values.csv"))


Total CSV files: 126
Reference columns: 84
[1/126] BenignTraffic.pcap_Flow.csv: 183,630 records
[2/126] BenignTraffic1.pcap_Flow.csv: 84,526 records
[3/126] BenignTraffic2.pcap_Flow.csv: 91,279 records
[4/126] BenignTraffic3.pcap_Flow.csv: 38,895 records
[5/126] DictionaryBruteForce.pcap_Flow.csv: 3,619 records
[6/126] DDoS-ACK_Fragmentation.pcap_Flow.csv: 323,886 records
[7/126] DDoS-ACK_Fragmentation1.pcap_Flow.csv: 137,172 records
[8/126] DDoS-ACK_Fragmentation12.pcap_Flow.csv: 37,292 records
[9/126] DDoS-ACK_Fragmentation2.pcap_Flow.csv: 121,409 records
[10/126] DDoS-ACK_Fragmentation3.pcap_Flow.csv: 174,443 records
[11/126] DDoS-ACK_Fragmentation4.pcap_Flow.csv: 152,216 records
[12/126] DDoS-ACK_Fragmentation5.pcap_Flow.csv: 209,344 records
[13/126] DDoS-ACK_Fragmentation6.pcap_Flow.csv: 209,790 records
[14/126] DDoS-ACK_Fragmentation7.pcap_Flow.csv: 127,543 records
[15/126] DDoS-ACK_Fragmentation8.pcap_Flow.csv: 179,379 records
[16/126] DDoS-ICMP_Flood.pcap_Flow.csv: 10,415 recor

,Feature,Missing_Count,Missing_Percentage
20,Flow Bytes/s,86310,0.320033
0,Flow ID,0,0.000000
2,Src Port,0,0.000000
1,Src IP,0,0.000000
4,Dst Port,0,0.000000
5,Protocol,0,0.000000
6,Timestamp,0,0.000000
7,Flow Duration,0,0.000000
8,Total Fwd Packet,0,0.000000
9,Total Bwd packets,0,0.000000



Report saved to:
/kaggle/working/O1_results/O1_DIAD_missing_values.csv


In [30]:

# ============================================================
# OBJECTIVE 1: EXACT DUPLICATE-RECORD ANALYSIS
# Dataset: CIC IoT-DIAD 2024
# Memory-efficient disk-based hash partitioning
# ============================================================

import os
import glob
import shutil
import numpy as np
import pandas as pd

DATASET_PATH = (
    "/kaggle/input/datasets/mostafaaliogab/"
    "cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024"
)

OUTPUT_DIR = "/kaggle/working/O1_results"
TEMP_DIR = "/kaggle/working/DIAD_duplicate_temp"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TEMP_DIR, exist_ok=True)

CHUNK_SIZE = 100000
N_PARTITIONS = 128

csv_files = glob.glob(
    os.path.join(DATASET_PATH, "**", "*.csv"),
    recursive=True
)

reference_file = next(
    (f for f in csv_files
     if os.path.basename(f) == "BenignTraffic.pcap_Flow.csv"),
    None
)

if reference_file is None:
    raise FileNotFoundError("Reference CSV not found.")

reference_columns = pd.read_csv(
    reference_file, nrows=0
).columns.tolist()

# Known headerless files from the previous schema audit
headerless_files = {
    f for f in csv_files
    if (
        "DoS-TCP_Flood" in f
        and os.path.basename(f).startswith("DoS-TCP_Flood")
    )
}

# Start with a clean temporary directory
if os.path.exists(TEMP_DIR):
    shutil.rmtree(TEMP_DIR)

os.makedirs(TEMP_DIR)

# ------------------------------------------------------------
# 1. Read records and write row hashes to disk partitions
# ------------------------------------------------------------

total_records = 0
failed_files = []

for i, file_path in enumerate(csv_files, 1):

    try:
        if file_path in headerless_files:
            reader = pd.read_csv(
                file_path,
                header=None,
                names=reference_columns,
                dtype=str,
                keep_default_na=False,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )
        else:
            reader = pd.read_csv(
                file_path,
                dtype=str,
                keep_default_na=False,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )

        for chunk in reader:

            # Normalize surrounding whitespace
            chunk = chunk.apply(
                lambda col: col.str.strip()
            )

            # Hash every column in each row
            row_hashes = pd.util.hash_pandas_object(
                chunk,
                index=False
            ).to_numpy(dtype=np.uint64)

            total_records += len(row_hashes)

            # Distribute hashes across disk partitions
            partition_ids = (
                row_hashes % np.uint64(N_PARTITIONS)
            ).astype(np.int16)

            for partition_id in np.unique(partition_ids):

                selected = row_hashes[
                    partition_ids == partition_id
                ]

                partition_file = os.path.join(
                    TEMP_DIR,
                    f"partition_{partition_id}.bin"
                )

                with open(partition_file, "ab") as f:
                    selected.tofile(f)

        print(
            f"[{i}/{len(csv_files)}] "
            f"Processed: {os.path.basename(file_path)}"
        )

    except Exception as e:
        failed_files.append({
            "file": file_path,
            "error": str(e)
        })
        print("ERROR:", file_path, str(e))

# ------------------------------------------------------------
# 2. Count repeated hashes in each partition
# ------------------------------------------------------------

duplicate_groups = 0
duplicate_records = 0
unique_hashes = 0

partition_results = []

for partition_id in range(N_PARTITIONS):

    partition_file = os.path.join(
        TEMP_DIR,
        f"partition_{partition_id}.bin"
    )

    if not os.path.exists(partition_file):
        continue

    hashes = np.fromfile(
        partition_file,
        dtype=np.uint64
    )

    if len(hashes) == 0:
        continue

    hashes.sort()

    values, counts = np.unique(
        hashes,
        return_counts=True
    )

    repeated = counts > 1

    groups = int(repeated.sum())
    extra_records = int((counts[repeated] - 1).sum())

    duplicate_groups += groups
    duplicate_records += extra_records
    unique_hashes += len(values)

    partition_results.append({
        "Partition": partition_id,
        "Records": len(hashes),
        "Unique_Hashes": len(values),
        "Duplicate_Groups": groups,
        "Duplicate_Records": extra_records
    })

# ------------------------------------------------------------
# 3. Summary and save reports
# ------------------------------------------------------------

duplicate_percentage = (
    duplicate_records / total_records * 100
    if total_records else 0
)

summary = pd.DataFrame([{
    "Total_Records_Processed": total_records,
    "Unique_Row_Hashes": unique_hashes,
    "Duplicate_Groups": duplicate_groups,
    "Duplicate_Records": duplicate_records,
    "Duplicate_Percentage": duplicate_percentage,
    "Failed_Files": len(failed_files)
}])

summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_duplicate_summary.csv"
    ),
    index=False
)

pd.DataFrame(partition_results).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_duplicate_partition_details.csv"
    ),
    index=False
)

if failed_files:
    pd.DataFrame(failed_files).to_csv(
        os.path.join(
            OUTPUT_DIR,
            "O1_DIAD_duplicate_failed_files.csv"
        ),
        index=False
    )

print("\n" + "=" * 65)
print("DUPLICATE-RECORD ANALYSIS SUMMARY")
print("=" * 65)

print("Files found:", len(csv_files))
print("Total records processed:", f"{total_records:,}")
print("Unique row hashes:", f"{unique_hashes:,}")
print("Duplicate groups:", f"{duplicate_groups:,}")
print("Duplicate records:", f"{duplicate_records:,}")
print(f"Duplicate percentage: {duplicate_percentage:.6f}%")
print("Failed files:", len(failed_files))

print("\nSaved summary:")
print(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_duplicate_summary.csv"
    )
)

# Remove temporary hash files after successful analysis
if not failed_files:
    shutil.rmtree(TEMP_DIR)
    print("\nTemporary hash files removed.")
else:
    print(
        "\nTemporary files retained because some files failed. "
        "Inspect the errors before interpreting the results."
    )


[1/126] Processed: BenignTraffic.pcap_Flow.csv
[2/126] Processed: BenignTraffic1.pcap_Flow.csv
[3/126] Processed: BenignTraffic2.pcap_Flow.csv
[4/126] Processed: BenignTraffic3.pcap_Flow.csv
[5/126] Processed: DictionaryBruteForce.pcap_Flow.csv
[6/126] Processed: DDoS-ACK_Fragmentation.pcap_Flow.csv
[7/126] Processed: DDoS-ACK_Fragmentation1.pcap_Flow.csv
[8/126] Processed: DDoS-ACK_Fragmentation12.pcap_Flow.csv
[9/126] Processed: DDoS-ACK_Fragmentation2.pcap_Flow.csv
[10/126] Processed: DDoS-ACK_Fragmentation3.pcap_Flow.csv
[11/126] Processed: DDoS-ACK_Fragmentation4.pcap_Flow.csv
[12/126] Processed: DDoS-ACK_Fragmentation5.pcap_Flow.csv
[13/126] Processed: DDoS-ACK_Fragmentation6.pcap_Flow.csv
[14/126] Processed: DDoS-ACK_Fragmentation7.pcap_Flow.csv
[15/126] Processed: DDoS-ACK_Fragmentation8.pcap_Flow.csv
[16/126] Processed: DDoS-ICMP_Flood.pcap_Flow.csv
[17/126] Processed: DDoS-ICMP_Flood1.pcap_Flow.csv
[18/126] Processed: DDoS-ICMP_Flood10.pcap_Flow.csv
[19/126] Processed: DDoS-I

In [31]:

# ============================================================
# OBJECTIVE 1: INFINITE AND INVALID NUMERICAL VALUE ANALYSIS
# Dataset: CIC IoT-DIAD 2024
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

DATASET_PATH = (
    "/kaggle/input/datasets/mostafaaliogab/"
    "cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024"
)

OUTPUT_DIR = "/kaggle/working/O1_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 100000

csv_files = glob.glob(
    os.path.join(DATASET_PATH, "**", "*.csv"),
    recursive=True
)

# Reference schema
reference_file = next(
    (f for f in csv_files
     if os.path.basename(f) == "BenignTraffic.pcap_Flow.csv"),
    None
)

if reference_file is None:
    raise FileNotFoundError("Reference CSV not found.")

reference_columns = pd.read_csv(
    reference_file, nrows=0
).columns.tolist()

# Exclude identifiers, timestamps and categorical metadata
excluded_names = {
    "flow id", "src ip", "dst ip", "timestamp",
    "label", "class", "attack", "attack type",
    "attack_type", "protocol"
}

numeric_candidates = [
    col for col in reference_columns
    if col.strip().lower() not in excluded_names
]

print("Total schema columns:", len(reference_columns))
print("Numerical candidates:", len(numeric_candidates))

# Known headerless files
headerless_files = {
    f for f in csv_files
    if (
        "DoS-TCP_Flood" in f
        and os.path.basename(f).startswith("DoS-TCP_Flood")
    )
}

# Initialize counters
stats = {
    col: {
        "Rows": 0,
        "Numeric_Values": 0,
        "Missing_Values": 0,
        "Invalid_Strings": 0,
        "Positive_Infinity": 0,
        "Negative_Infinity": 0,
        "Zero_Values": 0,
        "Negative_Values": 0
    }
    for col in numeric_candidates
}

total_records = 0
failed_files = []

# ------------------------------------------------------------
# Scan all files in chunks
# ------------------------------------------------------------

for i, file_path in enumerate(csv_files, 1):

    try:
        if file_path in headerless_files:
            reader = pd.read_csv(
                file_path,
                header=None,
                names=reference_columns,
                dtype=str,
                keep_default_na=False,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )
        else:
            reader = pd.read_csv(
                file_path,
                dtype=str,
                keep_default_na=False,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )

        for chunk in reader:

            total_records += len(chunk)

            for col in numeric_candidates:

                if col not in chunk.columns:
                    continue

                raw = chunk[col].astype(str).str.strip()

                # Empty values
                missing_mask = (
                    raw.eq("") |
                    raw.str.lower().isin(["nan", "null", "none"])
                )

                # Convert to numeric
                numeric = pd.to_numeric(
                    raw.mask(missing_mask),
                    errors="coerce"
                )

                numeric_array = numeric.to_numpy(
                    dtype=np.float64,
                    na_value=np.nan
                )

                invalid_mask = (
                    numeric.isna() & ~missing_mask
                )

                finite_mask = np.isfinite(numeric_array)

                stats[col]["Rows"] += len(raw)
                stats[col]["Missing_Values"] += int(
                    missing_mask.sum()
                )
                stats[col]["Invalid_Strings"] += int(
                    invalid_mask.sum()
                )

                stats[col]["Positive_Infinity"] += int(
                    np.isposinf(numeric_array).sum()
                )

                stats[col]["Negative_Infinity"] += int(
                    np.isneginf(numeric_array).sum()
                )

                stats[col]["Numeric_Values"] += int(
                    finite_mask.sum()
                )

                stats[col]["Zero_Values"] += int(
                    (finite_mask & (numeric_array == 0)).sum()
                )

                stats[col]["Negative_Values"] += int(
                    (finite_mask & (numeric_array < 0)).sum()
                )

        print(
            f"[{i}/{len(csv_files)}] "
            f"{os.path.basename(file_path)}"
        )

    except Exception as e:
        failed_files.append({
            "file": file_path,
            "error": str(e)
        })
        print("ERROR:", file_path, str(e))

# ------------------------------------------------------------
# Build report
# ------------------------------------------------------------

report = pd.DataFrame.from_dict(
    stats,
    orient="index"
).reset_index()

report = report.rename(
    columns={"index": "Feature"}
)

report["Positive_Infinity_Percentage"] = (
    report["Positive_Infinity"] /
    report["Rows"] * 100
)

report["Negative_Infinity_Percentage"] = (
    report["Negative_Infinity"] /
    report["Rows"] * 100
)

report["Invalid_String_Percentage"] = (
    report["Invalid_Strings"] /
    report["Rows"] * 100
)

report.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_numeric_quality_report.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("NUMERICAL DATA QUALITY SUMMARY")
print("=" * 65)

print("Files found:", len(csv_files))
print("Failed files:", len(failed_files))
print("Total records:", f"{total_records:,}")
print("Numerical candidate features:", len(numeric_candidates))

print(
    "Total positive infinity:",
    f"{report['Positive_Infinity'].sum():,}"
)

print(
    "Total negative infinity:",
    f"{report['Negative_Infinity'].sum():,}"
)

print(
    "Total invalid strings:",
    f"{report['Invalid_Strings'].sum():,}"
)

print(
    "Total zero values:",
    f"{report['Zero_Values'].sum():,}"
)

print(
    "Total negative values:",
    f"{report['Negative_Values'].sum():,}"
)

print("\nFEATURES WITH INFINITY OR INVALID STRINGS:")

problem_report = report[
    (report["Positive_Infinity"] > 0) |
    (report["Negative_Infinity"] > 0) |
    (report["Invalid_Strings"] > 0)
]

display(problem_report)

print("\nFLOW BYTES/S DETAILS:")

flow_bytes_report = report[
    report["Feature"].str.strip().str.lower()
    == "flow bytes/s"
]

display(flow_bytes_report)

if failed_files:
    print("\nFAILED FILES:")
    display(pd.DataFrame(failed_files))

print("\nReport saved:")
print(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_numeric_quality_report.csv"
    )
)


Total schema columns: 84
Numerical candidates: 78
[1/126] BenignTraffic.pcap_Flow.csv
[2/126] BenignTraffic1.pcap_Flow.csv
[3/126] BenignTraffic2.pcap_Flow.csv
[4/126] BenignTraffic3.pcap_Flow.csv
[5/126] DictionaryBruteForce.pcap_Flow.csv
[6/126] DDoS-ACK_Fragmentation.pcap_Flow.csv
[7/126] DDoS-ACK_Fragmentation1.pcap_Flow.csv
[8/126] DDoS-ACK_Fragmentation12.pcap_Flow.csv
[9/126] DDoS-ACK_Fragmentation2.pcap_Flow.csv
[10/126] DDoS-ACK_Fragmentation3.pcap_Flow.csv
[11/126] DDoS-ACK_Fragmentation4.pcap_Flow.csv
[12/126] DDoS-ACK_Fragmentation5.pcap_Flow.csv
[13/126] DDoS-ACK_Fragmentation6.pcap_Flow.csv
[14/126] DDoS-ACK_Fragmentation7.pcap_Flow.csv
[15/126] DDoS-ACK_Fragmentation8.pcap_Flow.csv
[16/126] DDoS-ICMP_Flood.pcap_Flow.csv
[17/126] DDoS-ICMP_Flood1.pcap_Flow.csv
[18/126] DDoS-ICMP_Flood10.pcap_Flow.csv
[19/126] DDoS-ICMP_Flood11.pcap_Flow.csv
[20/126] DDoS-ICMP_Flood12.pcap_Flow.csv
[21/126] DDoS-ICMP_Flood13.pcap_Flow.csv
[22/126] DDoS-ICMP_Flood14.pcap_Flow.csv
[23/126] D

,Feature,Rows,Numeric_Values,Missing_Values,Invalid_Strings,Positive_Infinity,Negative_Infinity,Zero_Values,Negative_Values,Positive_Infinity_Percentage,Negative_Infinity_Percentage,Invalid_String_Percentage
15,Flow Bytes/s,26969082,26879406,86310,0,3366,0,19128803,2,0.012481,0.0,0.0
16,Flow Packets/s,26969082,26879406,0,0,89676,0,0,2,0.332514,0.0,0.0



FLOW BYTES/S DETAILS:


,Feature,Rows,Numeric_Values,Missing_Values,Invalid_Strings,Positive_Infinity,Negative_Infinity,Zero_Values,Negative_Values,Positive_Infinity_Percentage,Negative_Infinity_Percentage,Invalid_String_Percentage
15,Flow Bytes/s,26969082,26879406,86310,0,3366,0,19128803,2,0.012481,0.0,0.0



Report saved:
/kaggle/working/O1_results/O1_DIAD_numeric_quality_report.csv


In [32]:

# ============================================================
# OBJECTIVE 1: INVESTIGATE NEGATIVE VALUES AND INFINITY
# Dataset: CIC IoT-DIAD 2024
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

DATASET_PATH = (
    "/kaggle/input/datasets/mostafaaliogab/"
    "cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024"
)

OUTPUT_DIR = "/kaggle/working/O1_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 100000

csv_files = glob.glob(
    os.path.join(DATASET_PATH, "**", "*.csv"),
    recursive=True
)

reference_file = next(
    (f for f in csv_files
     if os.path.basename(f) == "BenignTraffic.pcap_Flow.csv"),
    None
)

reference_columns = pd.read_csv(
    reference_file, nrows=0
).columns.tolist()

excluded_names = {
    "flow id", "src ip", "dst ip", "timestamp",
    "label", "class", "attack", "attack type",
    "attack_type", "protocol"
}

numeric_candidates = [
    col for col in reference_columns
    if col.strip().lower() not in excluded_names
]

headerless_files = {
    f for f in csv_files
    if (
        "DoS-TCP_Flood" in f
        and os.path.basename(f).startswith("DoS-TCP_Flood")
    )
}

# Counters
negative_counts = {col: 0 for col in numeric_candidates}

infinity_duration_counts = {
    "Flow Bytes/s": {
        "Total_Infinity": 0,
        "Duration_Zero": 0,
        "Duration_Negative": 0,
        "Duration_Positive": 0,
        "Duration_Missing": 0
    },
    "Flow Packets/s": {
        "Total_Infinity": 0,
        "Duration_Zero": 0,
        "Duration_Negative": 0,
        "Duration_Positive": 0,
        "Duration_Missing": 0
    }
}

negative_examples = []
infinity_examples = []

total_records = 0
failed_files = []

# ------------------------------------------------------------
# Scan all files
# ------------------------------------------------------------

for i, file_path in enumerate(csv_files, 1):

    try:
        if file_path in headerless_files:
            reader = pd.read_csv(
                file_path,
                header=None,
                names=reference_columns,
                dtype=str,
                keep_default_na=False,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )
        else:
            reader = pd.read_csv(
                file_path,
                dtype=str,
                keep_default_na=False,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )

        for chunk in reader:

            total_records += len(chunk)

            # Convert candidate features to numeric
            numeric_data = {}

            for col in numeric_candidates:
                if col in chunk.columns:
                    raw = chunk[col].astype(str).str.strip()

                    numeric_data[col] = pd.to_numeric(
                        raw.replace(
                            ["", "nan", "NaN", "NULL", "null"],
                            np.nan
                        ),
                        errors="coerce"
                    )

            numeric_df = pd.DataFrame(
                numeric_data,
                index=chunk.index
            )

            # Count negative values and retain examples
            for col in numeric_df.columns:

                mask = numeric_df[col] < 0
                count = int(mask.sum())

                negative_counts[col] += count

                if count > 0 and len(negative_examples) < 100:

                    selected = chunk.loc[mask].head(
                        100 - len(negative_examples)
                    )

                    for idx, row in selected.iterrows():
                        negative_examples.append({
                            "Feature": col,
                            "Value": row[col],
                            "File": os.path.basename(file_path),
                            "Flow_Duration": (
                                row["Flow Duration"]
                                if "Flow Duration" in row.index
                                else None
                            ),
                            "Flow_ID": (
                                row["Flow ID"]
                                if "Flow ID" in row.index
                                else None
                            )
                        })

            # Investigate infinity values in rate features
            if "Flow Duration" not in numeric_df.columns:
                continue

            duration = numeric_df["Flow Duration"]

            for rate_col in [
                "Flow Bytes/s",
                "Flow Packets/s"
            ]:

                if rate_col not in numeric_df.columns:
                    continue

                rate = numeric_df[rate_col]

                inf_mask = np.isinf(
                    rate.to_numpy(dtype=float)
                )

                inf_mask = pd.Series(
                    inf_mask,
                    index=chunk.index
                )

                count_inf = int(inf_mask.sum())

                if count_inf == 0:
                    continue

                result = infinity_duration_counts[rate_col]
                result["Total_Infinity"] += count_inf

                result["Duration_Zero"] += int(
                    (inf_mask & (duration == 0)).sum()
                )

                result["Duration_Negative"] += int(
                    (inf_mask & (duration < 0)).sum()
                )

                result["Duration_Positive"] += int(
                    (inf_mask & (duration > 0)).sum()
                )

                result["Duration_Missing"] += int(
                    (inf_mask & duration.isna()).sum()
                )

                if len(infinity_examples) < 50:

                    selected = chunk.loc[inf_mask].head(
                        50 - len(infinity_examples)
                    )

                    for idx, row in selected.iterrows():
                        infinity_examples.append({
                            "Rate_Feature": rate_col,
                            "Rate_Value": row[rate_col],
                            "Flow_Duration": row["Flow Duration"],
                            "File": os.path.basename(file_path),
                            "Flow_ID": (
                                row["Flow ID"]
                                if "Flow ID" in row.index
                                else None
                            )
                        })

        print(
            f"[{i}/{len(csv_files)}] "
            f"{os.path.basename(file_path)}"
        )

    except Exception as e:
        failed_files.append({
            "file": file_path,
            "error": str(e)
        })
        print("ERROR:", file_path, str(e))

# ------------------------------------------------------------
# Save negative-value report
# ------------------------------------------------------------

negative_report = pd.DataFrame(
    list(negative_counts.items()),
    columns=["Feature", "Negative_Count"]
)

negative_report = negative_report[
    negative_report["Negative_Count"] > 0
].sort_values("Negative_Count", ascending=False)

negative_report.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_negative_value_report.csv"
    ),
    index=False
)

pd.DataFrame(negative_examples).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_negative_value_examples.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Save infinity-duration report
# ------------------------------------------------------------

infinity_report = pd.DataFrame.from_dict(
    infinity_duration_counts,
    orient="index"
).reset_index().rename(
    columns={"index": "Rate_Feature"}
)

infinity_report.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_infinity_duration_report.csv"
    ),
    index=False
)

pd.DataFrame(infinity_examples).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_infinity_examples.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("NEGATIVE VALUE INVESTIGATION")
print("=" * 65)

print("Total records:", f"{total_records:,}")
print("Total negative values:", sum(negative_counts.values()))
print("Features containing negative values:", len(negative_report))

display(negative_report)

print("\nNEGATIVE VALUE EXAMPLES:")
display(pd.DataFrame(negative_examples))

print("\nINFINITY AND FLOW DURATION RELATIONSHIP:")
display(infinity_report)

print("\nINFINITY EXAMPLES:")
display(pd.DataFrame(infinity_examples).head(20))

print("\nFailed files:", len(failed_files))

print("\nAll investigation reports saved in:", OUTPUT_DIR)


[1/126] BenignTraffic.pcap_Flow.csv
[2/126] BenignTraffic1.pcap_Flow.csv
[3/126] BenignTraffic2.pcap_Flow.csv
[4/126] BenignTraffic3.pcap_Flow.csv
[5/126] DictionaryBruteForce.pcap_Flow.csv
[6/126] DDoS-ACK_Fragmentation.pcap_Flow.csv
[7/126] DDoS-ACK_Fragmentation1.pcap_Flow.csv
[8/126] DDoS-ACK_Fragmentation12.pcap_Flow.csv
[9/126] DDoS-ACK_Fragmentation2.pcap_Flow.csv
[10/126] DDoS-ACK_Fragmentation3.pcap_Flow.csv
[11/126] DDoS-ACK_Fragmentation4.pcap_Flow.csv
[12/126] DDoS-ACK_Fragmentation5.pcap_Flow.csv
[13/126] DDoS-ACK_Fragmentation6.pcap_Flow.csv
[14/126] DDoS-ACK_Fragmentation7.pcap_Flow.csv
[15/126] DDoS-ACK_Fragmentation8.pcap_Flow.csv
[16/126] DDoS-ICMP_Flood.pcap_Flow.csv
[17/126] DDoS-ICMP_Flood1.pcap_Flow.csv
[18/126] DDoS-ICMP_Flood10.pcap_Flow.csv
[19/126] DDoS-ICMP_Flood11.pcap_Flow.csv
[20/126] DDoS-ICMP_Flood12.pcap_Flow.csv
[21/126] DDoS-ICMP_Flood13.pcap_Flow.csv
[22/126] DDoS-ICMP_Flood14.pcap_Flow.csv
[23/126] DDoS-ICMP_Flood15.pcap_Flow.csv
[24/126] DDoS-ICMP_

,Feature,Negative_Count
20,Flow IAT Min,7
2,Flow Duration,2
15,Flow Bytes/s,2
16,Flow Packets/s,2
17,Flow IAT Mean,2
19,Flow IAT Max,2
30,Bwd IAT Min,2
25,Fwd IAT Min,1
26,Bwd IAT Total,1
27,Bwd IAT Mean,1



NEGATIVE VALUE EXAMPLES:


,Feature,Value,File,Flow_Duration,Flow_ID
0,Flow IAT Min,-5.0,DDoS-ICMP_Flood12.pcap_Flow.csv,93011240,192.168.137.125-192.168.137.238-0-0-0
1,Fwd IAT Min,-5.0,DDoS-ICMP_Flood12.pcap_Flow.csv,93011240,192.168.137.125-192.168.137.238-0-0-0
2,Flow IAT Min,-517965.0,DoS-UDP_Flood.pcap_Flow.csv,169362,192.168.137.158-157.249.81.141-60105-80-6
3,Flow IAT Min,-253085.0,DoS-UDP_Flood.pcap_Flow.csv,149280,192.168.137.158-157.249.81.141-46128-443-6
4,Flow IAT Min,-152307.0,DoS-UDP_Flood.pcap_Flow.csv,118291046,192.168.137.218-172.217.1.14-42117-443-17
5,Bwd IAT Total,-517964.0,DoS-UDP_Flood.pcap_Flow.csv,169362,192.168.137.158-157.249.81.141-60105-80-6
6,Bwd IAT Mean,-172654.6666666667,DoS-UDP_Flood.pcap_Flow.csv,169362,192.168.137.158-157.249.81.141-60105-80-6
7,Bwd IAT Min,-517965.0,DoS-UDP_Flood.pcap_Flow.csv,169362,192.168.137.158-157.249.81.141-60105-80-6
8,Bwd IAT Min,-152307.0,DoS-UDP_Flood.pcap_Flow.csv,118291046,192.168.137.218-172.217.1.14-42117-443-17
9,Flow IAT Min,-38346.0,DoS-UDP_Flood.pcap_Flow.csv,61822915,192.168.137.31-35.164.195.39-50093-8886-6



INFINITY AND FLOW DURATION RELATIONSHIP:


,Rate_Feature,Total_Infinity,Duration_Zero,Duration_Negative,Duration_Positive,Duration_Missing
0,Flow Bytes/s,3366,3366,0,0,0
1,Flow Packets/s,89676,89676,0,0,0



INFINITY EXAMPLES:


,Rate_Feature,Rate_Value,Flow_Duration,File,Flow_ID
0,Flow Bytes/s,Infinity,0,BenignTraffic.pcap_Flow.csv,192.168.137.83-13.225.196.108-56458-443-6
1,Flow Bytes/s,Infinity,0,BenignTraffic.pcap_Flow.csv,192.168.137.253-173.198.192.101-37008-4431-6
2,Flow Bytes/s,Infinity,0,BenignTraffic.pcap_Flow.csv,192.168.137.41-192.168.137.1-42811-53-17
3,Flow Bytes/s,Infinity,0,BenignTraffic.pcap_Flow.csv,192.168.137.83-13.225.196.108-56582-443-6
4,Flow Packets/s,Infinity,0,BenignTraffic.pcap_Flow.csv,13.225.196.108-192.168.137.83-443-56244-6
5,Flow Packets/s,Infinity,0,BenignTraffic.pcap_Flow.csv,13.225.196.108-192.168.137.172-443-33206-6
6,Flow Packets/s,Infinity,0,BenignTraffic.pcap_Flow.csv,192.168.137.83-13.225.196.108-56458-443-6
7,Flow Packets/s,Infinity,0,BenignTraffic.pcap_Flow.csv,192.168.137.253-173.198.192.101-37008-4431-6
8,Flow Packets/s,Infinity,0,BenignTraffic.pcap_Flow.csv,192.168.137.41-192.168.137.1-42811-53-17
9,Flow Packets/s,Infinity,0,BenignTraffic.pcap_Flow.csv,192.168.137.83-13.225.196.108-56582-443-6



Failed files: 0

All investigation reports saved in: /kaggle/working/O1_results


In [33]:

# ============================================================
# OBJECTIVE 1: CONSTANT AND HIGHLY CORRELATED FEATURES
# CIC IoT-DIAD 2024
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
from collections import defaultdict

DATASET_PATH = (
    "/kaggle/input/datasets/mostafaaliogab/"
    "cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024"
)

OUTPUT_DIR = "/kaggle/working/O1_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 100_000

# Find CSV files
files = sorted(
    glob.glob(os.path.join(DATASET_PATH, "**", "*.csv"), recursive=True)
)

print("CSV files found:", len(files))

# Reference schema
reference_file = next(
    (f for f in files if os.path.basename(f) == "BenignTraffic.pcap_Flow.csv"),
    None
)

if reference_file is None:
    raise FileNotFoundError("Reference schema file not found.")

reference_columns = pd.read_csv(reference_file, nrows=0).columns.tolist()

# Five known headerless files
headerless_files = {
    os.path.basename(f)
    for f in files
    if "DoS-TCP_Flood" in os.path.basename(f)
}

# Exclude identifiers, labels and non-numeric fields
excluded = {
    "Flow ID", "Src IP", "Dst IP", "Timestamp",
    "Label", "Class", "Attack", "Protocol",
    "Src Port", "Dst Port"
}

numeric_features = [
    c for c in reference_columns
    if c not in excluded
]

# Accumulate statistics
feature_stats = {
    c: {
        "count": 0,
        "missing": 0,
        "finite_count": 0,
        "minimum": np.inf,
        "maximum": -np.inf,
        "sum": 0.0,
        "sum_sq": 0.0
    }
    for c in numeric_features
}

# Pairwise correlation accumulator
pair_sums = defaultdict(lambda: np.zeros(6, dtype=np.float64))

total_rows = 0
failed_files = []

for file_index, file_path in enumerate(files, 1):

    filename = os.path.basename(file_path)
    print(f"[{file_index}/{len(files)}] {filename}")

    try:
        if filename in headerless_files:
            reader = pd.read_csv(
                file_path,
                header=None,
                names=reference_columns,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )
        else:
            reader = pd.read_csv(
                file_path,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )

        for chunk in reader:

            available = [
                c for c in numeric_features if c in chunk.columns
            ]

            numeric = chunk[available].apply(
                pd.to_numeric, errors="coerce"
            )

            total_rows += len(chunk)

            for col in available:

                values = numeric[col].to_numpy(dtype=np.float64)
                finite = values[np.isfinite(values)]

                st = feature_stats[col]
                st["count"] += len(values)
                st["missing"] += int(np.isnan(values).sum())
                st["finite_count"] += len(finite)

                if len(finite):
                    st["minimum"] = min(st["minimum"], finite.min())
                    st["maximum"] = max(st["maximum"], finite.max())
                    st["sum"] += finite.sum()
                    st["sum_sq"] += np.square(finite).sum()

            # Pairwise complete observations for correlation
            valid_cols = [
                c for c in available
                if np.isfinite(numeric[c].to_numpy(dtype=np.float64)).any()
            ]

            for i in range(len(valid_cols)):
                for j in range(i + 1, len(valid_cols)):

                    a = valid_cols[i]
                    b = valid_cols[j]

                    x = numeric[a].to_numpy(dtype=np.float64)
                    y = numeric[b].to_numpy(dtype=np.float64)

                    mask = np.isfinite(x) & np.isfinite(y)
                    if not mask.any():
                        continue

                    x = x[mask]
                    y = y[mask]

                    key = (a, b)
                    n = len(x)

                    pair_sums[key] += np.array([
                        n,
                        x.sum(),
                        y.sum(),
                        np.dot(x, x),
                        np.dot(y, y),
                        np.dot(x, y)
                    ], dtype=np.float64)

    except Exception as e:
        failed_files.append((filename, str(e)))
        print("ERROR:", str(e)[:250])

# ============================================================
# 1. FEATURE VARIANCE AND CONSTANT FEATURES
# ============================================================

rows = []

for col, st in feature_stats.items():

    n = st["finite_count"]

    if n == 0:
        mean = np.nan
        variance = np.nan
    else:
        mean = st["sum"] / n
        variance = max(
            0.0,
            (st["sum_sq"] / n) - (mean ** 2)
        )

    rows.append({
        "Feature": col,
        "Rows": st["count"],
        "Finite_Values": n,
        "Missing_Values": st["missing"],
        "Minimum": st["minimum"] if n else np.nan,
        "Maximum": st["maximum"] if n else np.nan,
        "Mean": mean,
        "Variance": variance,
        "Constant": bool(n > 0 and st["minimum"] == st["maximum"])
    })

feature_report = pd.DataFrame(rows)
feature_report = feature_report.sort_values(
    ["Constant", "Variance"],
    ascending=[False, True]
)

constant_report = feature_report[
    feature_report["Constant"]
].copy()

feature_report.to_csv(
    os.path.join(OUTPUT_DIR, "O1_DIAD_feature_variance_report.csv"),
    index=False
)

constant_report.to_csv(
    os.path.join(OUTPUT_DIR, "O1_DIAD_constant_features.csv"),
    index=False
)

# ============================================================
# 2. PAIRWISE CORRELATION
# ============================================================

correlation_rows = []

for (a, b), s in pair_sums.items():

    n, sx, sy, sxx, syy, sxy = s

    if n < 2:
        continue

    numerator = sxy - (sx * sy / n)
    denominator_x = sxx - (sx * sx / n)
    denominator_y = syy - (sy * sy / n)

    denominator = np.sqrt(
        max(0.0, denominator_x) * max(0.0, denominator_y)
    )

    if denominator == 0:
        corr = np.nan
    else:
        corr = numerator / denominator

    correlation_rows.append({
        "Feature_1": a,
        "Feature_2": b,
        "Pairwise_Rows": int(n),
        "Pearson_Correlation": corr,
        "Absolute_Correlation": abs(corr) if np.isfinite(corr) else np.nan
    })

correlation_report = pd.DataFrame(correlation_rows)

if not correlation_report.empty:
    correlation_report = correlation_report.sort_values(
        "Absolute_Correlation",
        ascending=False
    )

correlation_report.to_csv(
    os.path.join(OUTPUT_DIR, "O1_DIAD_correlation_report.csv"),
    index=False
)

high_corr = correlation_report[
    correlation_report["Absolute_Correlation"] >= 0.95
].copy()

high_corr.to_csv(
    os.path.join(OUTPUT_DIR, "O1_DIAD_high_correlation_pairs.csv"),
    index=False
)

# ============================================================
# 3. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 65)
print("FEATURE VARIATION AUDIT SUMMARY")
print("=" * 65)

print("Total rows processed:", f"{total_rows:,}")
print("Files processed:", len(files) - len(failed_files))
print("Failed files:", len(failed_files))
print("Numeric candidate features:", len(numeric_features))
print("Constant features:", len(constant_report))
print("Highly correlated pairs (|r| >= 0.95):", len(high_corr))

print("\nCONSTANT FEATURES:")
display(constant_report)

print("\nTOP 20 HIGHEST-CORRELATION PAIRS:")
display(correlation_report.head(20))

print("\nHIGHLY CORRELATED PAIRS (|r| >= 0.95):")
display(high_corr.head(50))

print("\nFAILED FILES:")
print(failed_files[:10])

print("\nReports saved in:", OUTPUT_DIR)


CSV files found: 126
[1/126] BenignTraffic.pcap_Flow.csv
[2/126] BenignTraffic1.pcap_Flow.csv
[3/126] BenignTraffic2.pcap_Flow.csv
[4/126] BenignTraffic3.pcap_Flow.csv
[5/126] DictionaryBruteForce.pcap_Flow.csv
[6/126] DDoS-ACK_Fragmentation.pcap_Flow.csv
[7/126] DDoS-ACK_Fragmentation1.pcap_Flow.csv
[8/126] DDoS-ACK_Fragmentation12.pcap_Flow.csv
[9/126] DDoS-ACK_Fragmentation2.pcap_Flow.csv
[10/126] DDoS-ACK_Fragmentation3.pcap_Flow.csv
[11/126] DDoS-ACK_Fragmentation4.pcap_Flow.csv
[12/126] DDoS-ACK_Fragmentation5.pcap_Flow.csv
[13/126] DDoS-ACK_Fragmentation6.pcap_Flow.csv
[14/126] DDoS-ACK_Fragmentation7.pcap_Flow.csv
[15/126] DDoS-ACK_Fragmentation8.pcap_Flow.csv
[16/126] DDoS-ICMP_Flood.pcap_Flow.csv
[17/126] DDoS-ICMP_Flood1.pcap_Flow.csv
[18/126] DDoS-ICMP_Flood10.pcap_Flow.csv
[19/126] DDoS-ICMP_Flood11.pcap_Flow.csv
[20/126] DDoS-ICMP_Flood12.pcap_Flow.csv
[21/126] DDoS-ICMP_Flood13.pcap_Flow.csv
[22/126] DDoS-ICMP_Flood14.pcap_Flow.csv
[23/126] DDoS-ICMP_Flood15.pcap_Flow.cs

,Feature,Rows,Finite_Values,Missing_Values,Minimum,Maximum,Mean,Variance,Constant
30,Bwd PSH Flags,26969082,26969082,0,0.0,0.0,0.0,0.0,True
31,Fwd URG Flags,26969082,26969082,0,0.0,0.0,0.0,0.0,True
32,Bwd URG Flags,26969082,26969082,0,0.0,0.0,0.0,0.0,True
47,URG Flag Count,26969082,26969082,0,0.0,0.0,0.0,0.0,True
54,Fwd Bytes/Bulk Avg,26969082,26969082,0,0.0,0.0,0.0,0.0,True
55,Fwd Packet/Bulk Avg,26969082,26969082,0,0.0,0.0,0.0,0.0,True
56,Fwd Bulk Rate Avg,26969082,26969082,0,0.0,0.0,0.0,0.0,True



TOP 20 HIGHEST-CORRELATION PAIRS:


,Feature_1,Feature_2,Pairwise_Rows,Pearson_Correlation,Absolute_Correlation
548,Fwd Packet Length Mean,Fwd Segment Size Avg,26969082,1.000000,1.000000
811,Bwd Packet Length Mean,Bwd Segment Size Avg,26969082,1.000000,1.000000
1195,Flow IAT Max,Idle Max,26969082,0.999443,0.999443
2195,Packet Length Mean,Average Packet Size,26969082,0.997068,0.997068
979,Flow Packets/s,Fwd Packets/s,26879406,0.996495,0.996495
2000,Bwd Header Length,ACK Flag Count,26969082,0.984785,0.984785
2846,Idle Mean,Idle Min,26969082,0.982775,0.982775
1312,Fwd IAT Mean,Fwd IAT Min,26969082,0.981756,0.981756
1022,Flow IAT Mean,Flow IAT Min,26969082,0.977317,0.977317
535,Fwd Packet Length Mean,Packet Length Mean,26969082,0.974304,0.974304



HIGHLY CORRELATED PAIRS (|r| >= 0.95):


,Feature_1,Feature_2,Pairwise_Rows,Pearson_Correlation,Absolute_Correlation
548,Fwd Packet Length Mean,Fwd Segment Size Avg,26969082,1.000000,1.000000
811,Bwd Packet Length Mean,Bwd Segment Size Avg,26969082,1.000000,1.000000
1195,Flow IAT Max,Idle Max,26969082,0.999443,0.999443
2195,Packet Length Mean,Average Packet Size,26969082,0.997068,0.997068
979,Flow Packets/s,Fwd Packets/s,26879406,0.996495,0.996495
2000,Bwd Header Length,ACK Flag Count,26969082,0.984785,0.984785
2846,Idle Mean,Idle Min,26969082,0.982775,0.982775
1312,Fwd IAT Mean,Fwd IAT Min,26969082,0.981756,0.981756
1022,Flow IAT Mean,Flow IAT Min,26969082,0.977317,0.977317
535,Fwd Packet Length Mean,Packet Length Mean,26969082,0.974304,0.974304



FAILED FILES:
[]

Reports saved in: /kaggle/working/O1_results


In [34]:

# ============================================================
# OBJECTIVE 1: VERIFY ORIGINAL DIAD LABEL VALUES
# ============================================================

import os
import glob
import pandas as pd
from collections import Counter

DATASET_PATH = (
    "/kaggle/input/datasets/mostafaaliogab/"
    "cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024"
)

CHUNK_SIZE = 100_000

files = sorted(
    glob.glob(os.path.join(DATASET_PATH, "**", "*.csv"), recursive=True)
)

reference_file = next(
    f for f in files
    if os.path.basename(f) == "BenignTraffic.pcap_Flow.csv"
)

reference_columns = pd.read_csv(reference_file, nrows=0).columns.tolist()

headerless_files = {
    os.path.basename(f)
    for f in files
    if "DoS-TCP_Flood" in os.path.basename(f)
}

# Find the original label column
label_candidates = [
    c for c in reference_columns
    if c.strip().lower() in ["label", "class", "attack", "attack_type"]
]

print("Possible original label columns:", label_candidates)

if not label_candidates:
    raise ValueError("No label column found in the reference schema.")

LABEL_COL = label_candidates[0]
print("Selected label column:", LABEL_COL)

# Count original label values
global_counts = Counter()
file_label_summary = []
failed_files = []

for i, file_path in enumerate(files, 1):

    filename = os.path.basename(file_path)
    file_counts = Counter()

    print(f"[{i}/{len(files)}] {filename}")

    try:
        if filename in headerless_files:
            reader = pd.read_csv(
                file_path,
                header=None,
                names=reference_columns,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )
        else:
            reader = pd.read_csv(
                file_path,
                chunksize=CHUNK_SIZE,
                low_memory=False
            )

        for chunk in reader:

            if LABEL_COL not in chunk.columns:
                raise ValueError(f"{LABEL_COL} column missing")

            values = (
                chunk[LABEL_COL]
                .fillna("<MISSING>")
                .astype(str)
                .str.strip()
            )

            counts = values.value_counts(dropna=False)

            for value, count in counts.items():
                file_counts[value] += int(count)
                global_counts[value] += int(count)

        file_label_summary.append({
            "File": filename,
            "Scenario_Folder": os.path.basename(
                os.path.dirname(file_path)
            ),
            "Original_Label_Values": " | ".join(
                sorted(file_counts.keys())
            ),
            "Number_of_Label_Values": len(file_counts),
            "Total_Rows": sum(file_counts.values())
        })

    except Exception as e:
        failed_files.append((filename, str(e)))
        print("ERROR:", str(e)[:200])

# Global label distribution
label_report = pd.DataFrame([
    {"Original_Label": label, "Record_Count": count}
    for label, count in global_counts.items()
]).sort_values("Record_Count", ascending=False)

file_report = pd.DataFrame(file_label_summary)

label_report.to_csv(
    "/kaggle/working/O1_results/O1_DIAD_original_label_distribution.csv",
    index=False
)

file_report.to_csv(
    "/kaggle/working/O1_results/O1_DIAD_original_labels_by_file.csv",
    index=False
)

print("\n" + "=" * 60)
print("ORIGINAL LABEL VERIFICATION SUMMARY")
print("=" * 60)

print("Files found:", len(files))
print("Files processed:", len(file_label_summary))
print("Failed files:", len(failed_files))
print("Original label column:", LABEL_COL)
print("Number of distinct original labels:", len(global_counts))
print("Total records counted:", sum(global_counts.values()))

print("\nORIGINAL LABEL DISTRIBUTION:")
display(label_report)

print("\nLABEL VALUES BY FILE (first 30):")
display(file_report.head(30))

print("\nFAILED FILES:")
print(failed_files)

print("\nReports saved in /kaggle/working/O1_results/")


Possible original label columns: ['Label']
Selected label column: Label
[1/126] BenignTraffic.pcap_Flow.csv
[2/126] BenignTraffic1.pcap_Flow.csv
[3/126] BenignTraffic2.pcap_Flow.csv
[4/126] BenignTraffic3.pcap_Flow.csv
[5/126] DictionaryBruteForce.pcap_Flow.csv
[6/126] DDoS-ACK_Fragmentation.pcap_Flow.csv
[7/126] DDoS-ACK_Fragmentation1.pcap_Flow.csv
[8/126] DDoS-ACK_Fragmentation12.pcap_Flow.csv
[9/126] DDoS-ACK_Fragmentation2.pcap_Flow.csv
[10/126] DDoS-ACK_Fragmentation3.pcap_Flow.csv
[11/126] DDoS-ACK_Fragmentation4.pcap_Flow.csv
[12/126] DDoS-ACK_Fragmentation5.pcap_Flow.csv
[13/126] DDoS-ACK_Fragmentation6.pcap_Flow.csv
[14/126] DDoS-ACK_Fragmentation7.pcap_Flow.csv
[15/126] DDoS-ACK_Fragmentation8.pcap_Flow.csv
[16/126] DDoS-ICMP_Flood.pcap_Flow.csv
[17/126] DDoS-ICMP_Flood1.pcap_Flow.csv
[18/126] DDoS-ICMP_Flood10.pcap_Flow.csv
[19/126] DDoS-ICMP_Flood11.pcap_Flow.csv
[20/126] DDoS-ICMP_Flood12.pcap_Flow.csv
[21/126] DDoS-ICMP_Flood13.pcap_Flow.csv
[22/126] DDoS-ICMP_Flood14.pc

,Original_Label,Record_Count
0,NeedManualLabel,26969082



LABEL VALUES BY FILE (first 30):


,File,Scenario_Folder,Original_Label_Values,Number_of_Label_Values,Total_Rows
0,BenignTraffic.pcap_Flow.csv,Benign,NeedManualLabel,1,183630
1,BenignTraffic1.pcap_Flow.csv,Benign,NeedManualLabel,1,84526
2,BenignTraffic2.pcap_Flow.csv,Benign,NeedManualLabel,1,91279
3,BenignTraffic3.pcap_Flow.csv,Benign,NeedManualLabel,1,38895
4,DictionaryBruteForce.pcap_Flow.csv,BruteForce,NeedManualLabel,1,3619
5,DDoS-ACK_Fragmentation.pcap_Flow.csv,DDoS ACK Fragmentation,NeedManualLabel,1,323886
6,DDoS-ACK_Fragmentation1.pcap_Flow.csv,DDoS ACK Fragmentation,NeedManualLabel,1,137172
7,DDoS-ACK_Fragmentation12.pcap_Flow.csv,DDoS ACK Fragmentation,NeedManualLabel,1,37292
8,DDoS-ACK_Fragmentation2.pcap_Flow.csv,DDoS ACK Fragmentation,NeedManualLabel,1,121409
9,DDoS-ACK_Fragmentation3.pcap_Flow.csv,DDoS ACK Fragmentation,NeedManualLabel,1,174443



FAILED FILES:
[]

Reports saved in /kaggle/working/O1_results/


In [35]:

import os

DATASET_PATH = (
    "/kaggle/input/datasets/mostafaaliogab/"
    "cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024"
)

print("DATASET ROOT:")
print(DATASET_PATH)

print("\nTOP-LEVEL CONTENTS:")
for item in sorted(os.listdir(DATASET_PATH)):
    full_path = os.path.join(DATASET_PATH, item)

    if os.path.isdir(full_path):
        print("[FOLDER]", item)
    else:
        print("[FILE]", item)

print("\nREADME FILES:")
for root, dirs, files in os.walk(DATASET_PATH):
    for filename in files:
        if filename.lower().startswith("readme"):
            path = os.path.join(root, filename)

            print("\n" + "=" * 70)
            print("README:", path)
            print("=" * 70)

            try:
                with open(path, "r", encoding="utf-8", errors="replace") as f:
                    print(f.read(12000))
            except Exception as e:
                print("Could not read:", e)


DATASET ROOT:
/kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024

TOP-LEVEL CONTENTS:
[FOLDER] Benign
[FOLDER] BruteForce
[FOLDER] DDos
[FOLDER] DoS
[FOLDER] Mirai
[FILE] README.txt.txt
[FOLDER] Recon
[FOLDER] Spoofing
[FOLDER] Web-Based

README FILES:

README: /kaggle/input/datasets/mostafaaliogab/cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024/README.txt.txt
The CICFlowMeter is an open source tool that generates Biflows from pcap files, and extracts features from these flows.



NOTE: TCP flows are usually terminated upon connection teardown (by FIN packet) while UDP flows are terminated by a flow timeout. The flow timeout value can be assigned arbitrarily by the individual scheme e.g., 600 seconds for both TCP and UDP.


NOTE:

The data found in this folder is comprised of 8 folders(Benign + 7 Attacks). Each folder will contain csv files for the kind of traffic specified by the folder name.

The data has not been labeled on a row basis.

But as per the indiv

In [36]:

import os
import pandas as pd

DATASET_PATH = (
    "/kaggle/input/datasets/mostafaaliogab/"
    "cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024"
)

OUTPUT_DIR = "/kaggle/working/O1_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Map folder names to standardized multiclass labels
FOLDER_TO_LABEL = {
    "Benign": "Benign",
    "BruteForce": "Brute Force",
    "DDos": "DDoS",
    "DoS": "DoS",
    "Mirai": "Mirai",
    "Recon": "Recon",
    "Spoofing": "Spoofing",
    "Web-Based": "Web-Based"
}

# Build a file-level mapping manifest
manifest = []
unknown_folders = []

for root, dirs, files in os.walk(DATASET_PATH):
    for filename in files:
        if not filename.lower().endswith(".csv"):
            continue

        folder = os.path.basename(root)

        if folder not in FOLDER_TO_LABEL:
            unknown_folders.append((folder, filename))
            continue

        manifest.append({
            "file_path": os.path.join(root, filename),
            "file_name": filename,
            "source_folder": folder,
            "scenario_label": FOLDER_TO_LABEL[folder]
        })

manifest_df = pd.DataFrame(manifest)

print("=" * 65)
print("DIAD SCENARIO-BASED LABEL MAPPING AUDIT")
print("=" * 65)

print("Total CSV files mapped:", len(manifest_df))
print("Unknown-folder files:", len(unknown_folders))

print("\nFILES PER SCENARIO LABEL:")
print(
    manifest_df["scenario_label"]
    .value_counts()
    .sort_index()
)

print("\nFOLDER-TO-LABEL MAPPING:")
print(
    manifest_df[
        ["source_folder", "scenario_label"]
    ].drop_duplicates().sort_values("source_folder").to_string(index=False)
)

if unknown_folders:
    print("\nUNKNOWN FOLDER FILES:")
    for item in unknown_folders:
        print(item)

# Save the mapping for reproducibility
manifest_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_scenario_label_mapping.csv"
)

manifest_df.to_csv(manifest_path, index=False)

print("\nMapping manifest saved to:")
print(manifest_path)

# Validation
expected_folders = set(FOLDER_TO_LABEL.keys())
observed_folders = set(manifest_df["source_folder"].unique())

print("\nVALIDATION:")
print("All 8 expected folders represented:",
      expected_folders == observed_folders)
print("Exactly 126 CSV files mapped:",
      len(manifest_df) == 126)
print("No unknown-folder files:",
      len(unknown_folders) == 0)


DIAD SCENARIO-BASED LABEL MAPPING AUDIT
Total CSV files mapped: 34
Unknown-folder files: 92

FILES PER SCENARIO LABEL:
scenario_label
Benign          4
Brute Force     1
Mirai          29
Name: count, dtype: int64

FOLDER-TO-LABEL MAPPING:
source_folder scenario_label
       Benign         Benign
   BruteForce    Brute Force
        Mirai          Mirai

UNKNOWN FOLDER FILES:
('DDoS ACK Fragmentation', 'DDoS-ACK_Fragmentation.pcap_Flow.csv')
('DDoS ACK Fragmentation', 'DDoS-ACK_Fragmentation1.pcap_Flow.csv')
('DDoS ACK Fragmentation', 'DDoS-ACK_Fragmentation12.pcap_Flow.csv')
('DDoS ACK Fragmentation', 'DDoS-ACK_Fragmentation2.pcap_Flow.csv')
('DDoS ACK Fragmentation', 'DDoS-ACK_Fragmentation3.pcap_Flow.csv')
('DDoS ACK Fragmentation', 'DDoS-ACK_Fragmentation4.pcap_Flow.csv')
('DDoS ACK Fragmentation', 'DDoS-ACK_Fragmentation5.pcap_Flow.csv')
('DDoS ACK Fragmentation', 'DDoS-ACK_Fragmentation6.pcap_Flow.csv')
('DDoS ACK Fragmentation', 'DDoS-ACK_Fragmentation7.pcap_Flow.csv')
('DDoS AC

In [37]:

import os
import pandas as pd

DATASET_PATH = (
    "/kaggle/input/datasets/mostafaaliogab/"
    "cic-iot-diad-2024-dataset/CIC-IoT-DIAD2024"
)

OUTPUT_DIR = "/kaggle/working/O1_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TOP_FOLDER_TO_LABEL = {
    "Benign": "Benign",
    "BruteForce": "Brute Force",
    "DDos": "DDoS",
    "DoS": "DoS",
    "Mirai": "Mirai",
    "Recon": "Recon",
    "Spoofing": "Spoofing",
    "Web-Based": "Web-Based"
}

manifest = []
unknown_files = []

for root, dirs, files in os.walk(DATASET_PATH):
    for filename in files:
        if not filename.lower().endswith(".csv"):
            continue

        relative_path = os.path.relpath(root, DATASET_PATH)
        path_parts = relative_path.split(os.sep)

        # The first folder below the dataset root is the attack category
        top_folder = path_parts[0]

        # The immediate folder containing the CSV is the scenario
        scenario_folder = path_parts[-1]

        if top_folder not in TOP_FOLDER_TO_LABEL:
            unknown_files.append({
                "file": filename,
                "relative_folder": relative_path
            })
            continue

        manifest.append({
            "file_path": os.path.join(root, filename),
            "file_name": filename,
            "top_level_folder": top_folder,
            "scenario_folder": scenario_folder,
            "scenario_label": TOP_FOLDER_TO_LABEL[top_folder]
        })

manifest_df = pd.DataFrame(manifest)

print("=" * 65)
print("CORRECTED DIAD LABEL MAPPING AUDIT")
print("=" * 65)

print("Mapped CSV files:", len(manifest_df))
print("Unknown CSV files:", len(unknown_files))

print("\nFILES PER MULTICLASS LABEL:")
print(
    manifest_df["scenario_label"]
    .value_counts()
    .sort_index()
)

print("\nFILES PER TOP-LEVEL FOLDER:")
print(
    manifest_df["top_level_folder"]
    .value_counts()
    .sort_index()
)

print("\nSCENARIOS WITHIN EACH CLASS:")
print(
    manifest_df.groupby(
        ["scenario_label", "scenario_folder"]
    ).size().to_string()
)

print("\nVALIDATION:")
print(
    "All 126 files mapped:",
    len(manifest_df) == 126
)
print(
    "All 8 classes represented:",
    manifest_df["scenario_label"].nunique() == 8
)
print(
    "No unknown files:",
    len(unknown_files) == 0
)

if unknown_files:
    print("\nUNKNOWN FILES:")
    print(pd.DataFrame(unknown_files).to_string(index=False))

manifest_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_scenario_label_mapping_corrected.csv"
)

manifest_df.to_csv(manifest_path, index=False)

print("\nManifest saved:", manifest_path)


CORRECTED DIAD LABEL MAPPING AUDIT
Mapped CSV files: 126
Unknown CSV files: 0

FILES PER MULTICLASS LABEL:
scenario_label
Benign          4
Brute Force     1
DDoS           55
DoS            30
Mirai          29
Recon           1
Spoofing        3
Web-Based       3
Name: count, dtype: int64

FILES PER TOP-LEVEL FOLDER:
top_level_folder
Benign         4
BruteForce     1
DDos          55
DoS           30
Mirai         29
Recon          1
Spoofing       3
Web-Based      3
Name: count, dtype: int64

SCENARIOS WITHIN EACH CLASS:
scenario_label  scenario_folder        
Benign          Benign                      4
Brute Force     BruteForce                  1
DDoS            DDoS ACK Fragmentation     10
                DDoS ICMP Flood            27
                DDoS-HTTP Flood             1
                DDoS-ICMP_Fragmentation    17
DoS             DoS SYN Flood               8
                DoS-HTTP_Flood              2
                DoS-TCP_Flood               5
                

In [38]:

import os
import pandas as pd

OUTPUT_DIR = "/kaggle/working/O1_results"

manifest_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_scenario_label_mapping_corrected.csv"
)

manifest_df = pd.read_csv(manifest_path)

CHUNK_SIZE = 200_000

class_record_counts = {}
file_record_counts = []

for i, row in manifest_df.iterrows():

    path = row["file_path"]
    label = row["scenario_label"]

    # Detect whether the CSV has a header
    columns = pd.read_csv(path, nrows=0).columns.tolist()

    if "Flow ID" in columns:
        reader = pd.read_csv(path, chunksize=CHUNK_SIZE)
    else:
        # The five previously identified headerless files
        reader = pd.read_csv(
            path,
            header=None,
            chunksize=CHUNK_SIZE
        )

    file_count = 0

    for chunk in reader:
        file_count += len(chunk)

    class_record_counts[label] = (
        class_record_counts.get(label, 0) + file_count
    )

    file_record_counts.append({
        "file_name": row["file_name"],
        "scenario_label": label,
        "scenario_folder": row["scenario_folder"],
        "record_count": file_count
    })

    if (i + 1) % 10 == 0 or i + 1 == len(manifest_df):
        print(f"Processed {i + 1}/{len(manifest_df)} files")

# Create class-level record distribution
distribution_df = pd.DataFrame([
    {
        "class": label,
        "records": count
    }
    for label, count in class_record_counts.items()
])

distribution_df["percentage"] = (
    distribution_df["records"] /
    distribution_df["records"].sum() * 100
)

distribution_df = distribution_df.sort_values(
    "records", ascending=False
).reset_index(drop=True)

# Calculate imbalance ratio
largest_class = distribution_df["records"].max()
smallest_class = distribution_df["records"].min()

imbalance_ratio = largest_class / smallest_class

# Save results
distribution_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_scenario_record_distribution.csv"
    ),
    index=False
)

file_counts_df = pd.DataFrame(file_record_counts)

file_counts_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_scenario_record_counts_by_file.csv"
    ),
    index=False
)

print("\n" + "=" * 65)
print("DIAD RECORD-LEVEL SCENARIO DISTRIBUTION")
print("=" * 65)

print(distribution_df.to_string(index=False))

print("\nTOTAL RECORDS:", distribution_df["records"].sum())
print("NUMBER OF CLASSES:", len(distribution_df))
print("IMBALANCE RATIO:", round(imbalance_ratio, 2))

print("\nCLASS DISTRIBUTION VALIDATION:")
print(
    "Matches previous total of 26,969,082:",
    distribution_df["records"].sum() == 26_969_082
)

print("\nReports saved in:", OUTPUT_DIR)


Processed 10/126 files
Processed 20/126 files
Processed 30/126 files
Processed 40/126 files
Processed 50/126 files
Processed 60/126 files
Processed 70/126 files
Processed 80/126 files
Processed 90/126 files
Processed 100/126 files
Processed 110/126 files
Processed 120/126 files
Processed 126/126 files

DIAD RECORD-LEVEL SCENARIO DISTRIBUTION
      class  records  percentage
        DoS 23143929   85.816525
       DDoS  2637892    9.781171
      Recon   442158    1.639500
     Benign   398330    1.476988
      Mirai   174588    0.647364
   Spoofing   157238    0.583031
  Web-Based    11328    0.042004
Brute Force     3619    0.013419

TOTAL RECORDS: 26969082
NUMBER OF CLASSES: 8
IMBALANCE RATIO: 6395.12

CLASS DISTRIBUTION VALIDATION:
Matches previous total of 26,969,082: True

Reports saved in: /kaggle/working/O1_results


In [39]:

import os
import pandas as pd

OUTPUT_DIR = "/kaggle/working/O1_results"

file_counts_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_scenario_record_counts_by_file.csv"
)

file_df = pd.read_csv(file_counts_path)

# Summarize record concentration by class
summary = (
    file_df.groupby("scenario_label")
    .agg(
        number_of_files=("file_name", "nunique"),
        total_records=("record_count", "sum"),
        smallest_file=("record_count", "min"),
        largest_file=("record_count", "max"),
        mean_records_per_file=("record_count", "mean")
    )
    .reset_index()
)

summary["percentage_of_dataset"] = (
    summary["total_records"] /
    summary["total_records"].sum() * 100
)

summary = summary.sort_values(
    "total_records", ascending=False
)

print("=" * 75)
print("CLASS-LEVEL FILE CONCENTRATION ANALYSIS")
print("=" * 75)

print(summary.to_string(index=False))

print("\n" + "=" * 75)
print("RARE CLASS FILE DETAILS")
print("=" * 75)

rare_classes = ["Brute Force", "Web-Based", "Spoofing", "Mirai", "Recon"]

for label in rare_classes:
    print(f"\n{label}:")
    subset = file_df[file_df["scenario_label"] == label]

    print(
        subset[
            ["file_name", "scenario_folder", "record_count"]
        ].to_string(index=False)
    )

summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_class_file_concentration.csv"
    ),
    index=False
)

print("\nReport saved:")
print(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_class_file_concentration.csv"
    )
)


CLASS-LEVEL FILE CONCENTRATION ANALYSIS
scenario_label  number_of_files  total_records  smallest_file  largest_file  mean_records_per_file  percentage_of_dataset
           DoS               30       23143929              1       3990181          771464.300000              85.816525
          DDoS               55        2637892           3244        505720           47961.672727               9.781171
         Recon                1         442158         442158        442158          442158.000000               1.639500
        Benign                4         398330          38895        183630           99582.500000               1.476988
         Mirai               29         174588           2500         12079            6020.275862               0.647364
      Spoofing                3         157238           9349         74856           52412.666667               0.583031
     Web-Based                3          11328           1348          6603            3776.000000        

In [40]:

import os
import pandas as pd

OUTPUT_DIR = "/kaggle/working/O1_results"

manifest = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_scenario_label_mapping_corrected.csv"
    )
)

# Select up to 2 files from each class
sample_files = (
    manifest.groupby("scenario_label", group_keys=False)
    .head(2)
    .reset_index(drop=True)
)

results = []

for _, row in sample_files.iterrows():

    path = row["file_path"]

    try:
        df = pd.read_csv(path, usecols=["Timestamp"])

        timestamps = pd.to_datetime(
            df["Timestamp"],
            errors="coerce"
        )

        valid = timestamps.notna()

        results.append({
            "class": row["scenario_label"],
            "file": row["file_name"],
            "records": len(df),
            "valid_timestamps": int(valid.sum()),
            "invalid_timestamps": int((~valid).sum()),
            "valid_percentage": round(valid.mean() * 100, 4),
            "earliest_timestamp": (
                str(timestamps[valid].min()) if valid.any() else None
            ),
            "latest_timestamp": (
                str(timestamps[valid].max()) if valid.any() else None
            ),
            "timestamps_sorted": bool(
                timestamps[valid].is_monotonic_increasing
            )
        })

    except Exception as e:
        results.append({
            "class": row["scenario_label"],
            "file": row["file_name"],
            "error": str(e)
        })

result_df = pd.DataFrame(results)

print("=" * 75)
print("DIAD TIMESTAMP QUALITY SAMPLE AUDIT")
print("=" * 75)
print(result_df.to_string(index=False))

result_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_timestamp_sample_audit.csv"
    ),
    index=False
)

print("\nSaved timestamp sample audit.")


/tmp/ipykernel_23/3097942825.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamps = pd.to_datetime(
/tmp/ipykernel_23/3097942825.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamps = pd.to_datetime(
/tmp/ipykernel_23/3097942825.py:29: UserWarning: Parsing dates in %d/%m/%Y %I:%M:%S %p format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  timestamps = pd.to_datetime(
/tmp/ipykernel_23/3097942825.py:29: UserWarning: Parsing dates in %d/%m/%Y %I:%M:%S %p format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  timestamps = pd.to_datetime(
/tmp/ipykernel_23/

DIAD TIMESTAMP QUALITY SAMPLE AUDIT
      class                                  file  records  valid_timestamps  invalid_timestamps  valid_percentage  earliest_timestamp    latest_timestamp  timestamps_sorted
     Benign           BenignTraffic.pcap_Flow.csv   183630            183630                   0             100.0 2022-07-10 14:15:00 2022-07-10 23:30:27              False
     Benign          BenignTraffic1.pcap_Flow.csv    84526             84526                   0             100.0 2022-07-10 23:30:28 2022-08-10 06:45:20              False
Brute Force    DictionaryBruteForce.pcap_Flow.csv     3619              3619                   0             100.0 2022-10-14 13:18:01 2022-10-17 12:02:00              False
       DDoS  DDoS-ACK_Fragmentation.pcap_Flow.csv   323886            323886                   0             100.0 2022-10-25 09:26:00 2022-10-25 10:20:19              False
       DDoS DDoS-ACK_Fragmentation1.pcap_Flow.csv   137172            137172                  

/tmp/ipykernel_23/3097942825.py:29: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  timestamps = pd.to_datetime(
/tmp/ipykernel_23/3097942825.py:29: UserWarning: Parsing dates in %d/%m/%Y %I:%M:%S %p format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  timestamps = pd.to_datetime(
/tmp/ipykernel_23/3097942825.py:29: UserWarning: Parsing dates in %d/%m/%Y %I:%M:%S %p format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  timestamps = pd.to_datetime(


In [41]:

import os
import pandas as pd

OUTPUT_DIR = "/kaggle/working/O1_results"

manifest = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_scenario_label_mapping_corrected.csv"
    )
)

CHUNK_SIZE = 200_000
results = []

for i, row in manifest.iterrows():

    path = row["file_path"]

    earliest = None
    latest = None
    total = 0
    valid_count = 0
    invalid_count = 0
    sorted_within_chunks = True
    previous_max = None

    try:
        reader = pd.read_csv(
            path,
            usecols=["Timestamp"],
            chunksize=CHUNK_SIZE
        )

        for chunk in reader:

            raw = chunk["Timestamp"].astype(str).str.strip()

            # Parse both common formats explicitly
            parsed = pd.to_datetime(
                raw,
                format="%m/%d/%Y %I:%M:%S %p",
                errors="coerce"
            )

            unresolved = parsed.isna()

            if unresolved.any():
                parsed_alt = pd.to_datetime(
                    raw[unresolved],
                    format="%d/%m/%Y %I:%M:%S %p",
                    errors="coerce"
                )
                parsed.loc[unresolved] = parsed_alt

            # Try general parsing only for remaining values
            unresolved = parsed.isna()

            if unresolved.any():
                parsed_alt = pd.to_datetime(
                    raw[unresolved],
                    errors="coerce",
                    dayfirst=True
                )
                parsed.loc[unresolved] = parsed_alt

            valid = parsed.dropna()

            total += len(raw)
            valid_count += len(valid)
            invalid_count += parsed.isna().sum()

            if len(valid):
                chunk_min = valid.min()
                chunk_max = valid.max()

                earliest = (
                    chunk_min if earliest is None
                    else min(earliest, chunk_min)
                )
                latest = (
                    chunk_max if latest is None
                    else max(latest, chunk_max)
                )

                if previous_max is not None:
                    if chunk_min < previous_max:
                        sorted_within_chunks = False

                if not valid.is_monotonic_increasing:
                    sorted_within_chunks = False

                previous_max = chunk_max

        results.append({
            "class": row["scenario_label"],
            "file": row["file_name"],
            "records": total,
            "valid_timestamps": valid_count,
            "invalid_timestamps": invalid_count,
            "valid_percentage": round(
                valid_count / total * 100, 4
            ) if total else 0,
            "earliest_timestamp": earliest,
            "latest_timestamp": latest,
            "chronologically_sorted": sorted_within_chunks
        })

    except Exception as e:
        results.append({
            "class": row["scenario_label"],
            "file": row["file_name"],
            "error": str(e)
        })

    if (i + 1) % 10 == 0 or i + 1 == len(manifest):
        print(f"Processed {i + 1}/{len(manifest)} files")

timestamp_df = pd.DataFrame(results)

output_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_timestamp_full_audit.csv"
)

timestamp_df.to_csv(output_path, index=False)

print("\n" + "=" * 75)
print("FULL DIAD TIMESTAMP AUDIT")
print("=" * 75)

print("Files audited:", len(timestamp_df))
print(
    "Files with errors:",
    timestamp_df.get("error", pd.Series(dtype=object)).notna().sum()
)

print(
    "Total valid timestamps:",
    timestamp_df["valid_timestamps"].sum()
)

print(
    "Total invalid timestamps:",
    timestamp_df["invalid_timestamps"].sum()
)

print("\nTIMESTAMP RANGE BY CLASS:")

print(
    timestamp_df.groupby("class").agg(
        files=("file", "count"),
        earliest=("earliest_timestamp", "min"),
        latest=("latest_timestamp", "max")
    ).to_string()
)

print("\nFiles already chronologically sorted:",
      timestamp_df["chronologically_sorted"].sum())

print("\nSaved:", output_path)


Processed 10/126 files
Processed 20/126 files
Processed 30/126 files
Processed 40/126 files
Processed 50/126 files
Processed 60/126 files
Processed 70/126 files
Processed 80/126 files
Processed 90/126 files
Processed 100/126 files
Processed 110/126 files
Processed 120/126 files
Processed 126/126 files

FULL DIAD TIMESTAMP AUDIT
Files audited: 126
Files with errors: 5
Total valid timestamps: 26969077.0
Total invalid timestamps: 0.0

TIMESTAMP RANGE BY CLASS:
             files            earliest              latest
class                                                     
Benign           4 2022-07-10 14:15:00 2022-08-10 18:15:00
Brute Force      1 2022-10-14 13:18:01 2022-10-17 12:02:00
DDoS            55 2022-04-10 13:55:00 2022-12-09 17:25:17
DoS             30 2022-02-08 10:42:00 2022-12-08 17:10:00
Mirai           29 2023-01-13 10:22:00 2023-12-01 15:49:00
Recon            1 2022-03-10 11:02:00 2022-04-10 09:31:03
Spoofing         3 2022-10-17 17:20:00 2022-11-16 16:25:00
Web-Bas

In [42]:

# ============================================================
# O1: DIAD TIMESTAMP AUDIT - ERROR AND ORDERING DIAGNOSTICS
# ============================================================

import os
import pandas as pd

OUTPUT_DIR = "/kaggle/working/O1_results"

audit_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_timestamp_full_audit.csv"
)

timestamp_df = pd.read_csv(audit_path)

# ------------------------------------------------------------
# 1. IDENTIFY FILES WITH ERRORS
# ------------------------------------------------------------

print("=" * 75)
print("FILES WITH TIMESTAMP AUDIT ERRORS")
print("=" * 75)

if "error" in timestamp_df.columns:

    error_files = timestamp_df[
        timestamp_df["error"].notna()
    ]

    print("Number of error files:", len(error_files))

    if len(error_files):
        print(
            error_files[
                ["class", "file", "error"]
            ].to_string(index=False)
        )
else:
    error_files = pd.DataFrame()
    print("No error column found.")

# ------------------------------------------------------------
# 2. IDENTIFY FILES NOT CHRONOLOGICALLY SORTED
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FILES NOT CHRONOLOGICALLY SORTED")
print("=" * 75)

valid_audits = timestamp_df[
    timestamp_df["error"].isna()
].copy()

valid_audits["chronologically_sorted"] = (
    valid_audits["chronologically_sorted"]
    .astype(str)
    .str.lower()
    .map({"true": True, "false": False})
)

unsorted_files = valid_audits[
    valid_audits["chronologically_sorted"] == False
]

print("Successfully audited files:", len(valid_audits))
print("Unsorted files:", len(unsorted_files))

print(
    unsorted_files[
        ["class", "file", "records",
         "earliest_timestamp", "latest_timestamp"]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 3. SUMMARY BY CLASS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("CHRONOLOGICAL ORDER SUMMARY BY CLASS")
print("=" * 75)

print(
    valid_audits.groupby("class").agg(
        files=("file", "count"),
        sorted_files=("chronologically_sorted", "sum"),
        unsorted_files=("chronologically_sorted",
                        lambda x: (~x).sum())
    ).to_string()
)

print("\nDiagnostic completed.")


FILES WITH TIMESTAMP AUDIT ERRORS
Number of error files: 5
class                          file                                                                       error
  DoS   DoS-TCP_Flood.pcap_Flow.csv Usecols do not match columns, columns expected but not found: ['Timestamp']
  DoS  DoS-TCP_Flood1.pcap_Flow.csv Usecols do not match columns, columns expected but not found: ['Timestamp']
  DoS DoS-TCP_Flood10.pcap_Flow.csv Usecols do not match columns, columns expected but not found: ['Timestamp']
  DoS  DoS-TCP_Flood2.pcap_Flow.csv Usecols do not match columns, columns expected but not found: ['Timestamp']
  DoS  DoS-TCP_Flood3.pcap_Flow.csv Usecols do not match columns, columns expected but not found: ['Timestamp']

FILES NOT CHRONOLOGICALLY SORTED
Successfully audited files: 121
Unsorted files: 121
      class                                    file   records  earliest_timestamp    latest_timestamp
     Benign             BenignTraffic.pcap_Flow.csv  183630.0 2022-07-10 14:15:00

In [43]:

# ============================================================
# O1: DIAD CORRECTED TIMESTAMP ORDERING AUDIT
# All 126 files | Header-aware | Chunk-based
# ============================================================

import os
import pandas as pd

OUTPUT_DIR = "/kaggle/working/O1_results"
MANIFEST_PATH = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_scenario_label_mapping_corrected.csv"
)

CHUNK_SIZE = 200_000

manifest = pd.read_csv(MANIFEST_PATH)

# ------------------------------------------------------------
# 1. OBTAIN VERIFIED REFERENCE SCHEMA
# ------------------------------------------------------------

reference_path = manifest.loc[
    manifest["file_name"] == "BenignTraffic.pcap_Flow.csv",
    "file_path"
].iloc[0]

reference_columns = pd.read_csv(
    reference_path,
    nrows=0
).columns.tolist()

print("Reference columns:", len(reference_columns))

# ------------------------------------------------------------
# 2. KNOWN HEADERLESS FILES
# ------------------------------------------------------------

headerless_files = {
    "DoS-TCP_Flood.pcap_Flow.csv",
    "DoS-TCP_Flood1.pcap_Flow.csv",
    "DoS-TCP_Flood2.pcap_Flow.csv",
    "DoS-TCP_Flood3.pcap_Flow.csv",
    "DoS-TCP_Flood10.pcap_Flow.csv"
}

# ------------------------------------------------------------
# 3. TIMESTAMP PARSER
# ------------------------------------------------------------

def parse_timestamps(raw):

    raw = raw.astype(str).str.strip()

    parsed = pd.to_datetime(
        raw,
        format="%m/%d/%Y %I:%M:%S %p",
        errors="coerce"
    )

    unresolved = parsed.isna()

    if unresolved.any():
        parsed_alt = pd.to_datetime(
            raw[unresolved],
            format="%d/%m/%Y %I:%M:%S %p",
            errors="coerce"
        )
        parsed.loc[unresolved] = parsed_alt

    unresolved = parsed.isna()

    if unresolved.any():
        parsed_alt = pd.to_datetime(
            raw[unresolved],
            errors="coerce",
            dayfirst=True
        )
        parsed.loc[unresolved] = parsed_alt

    return parsed

# ------------------------------------------------------------
# 4. AUDIT ALL FILES
# ------------------------------------------------------------

results = []

for i, row in manifest.iterrows():

    path = row["file_path"]
    filename = row["file_name"]

    records = 0
    valid_count = 0
    invalid_count = 0
    reversal_count = 0

    earliest = None
    latest = None
    previous_timestamp = None
    first_violation = None

    try:

        if filename in headerless_files:

            reader = pd.read_csv(
                path,
                header=None,
                names=reference_columns,
                usecols=["Timestamp"],
                chunksize=CHUNK_SIZE
            )

        else:

            reader = pd.read_csv(
                path,
                usecols=["Timestamp"],
                chunksize=CHUNK_SIZE
            )

        for chunk in reader:

            parsed = parse_timestamps(
                chunk["Timestamp"]
            )

            records += len(parsed)
            valid_count += parsed.notna().sum()
            invalid_count += parsed.isna().sum()

            valid = parsed.dropna()

            if len(valid) == 0:
                continue

            chunk_min = valid.min()
            chunk_max = valid.max()

            earliest = (
                chunk_min if earliest is None
                else min(earliest, chunk_min)
            )

            latest = (
                chunk_max if latest is None
                else max(latest, chunk_max)
            )

            # Count adjacent timestamp reversals
            previous_values = valid.shift(1)

            inversions = (
                valid < previous_values
            )

            inversions.iloc[0] = False

            current_reversals = int(inversions.sum())

            # Check across chunk boundaries
            if previous_timestamp is not None:

                if valid.iloc[0] < previous_timestamp:
                    current_reversals += 1

                    if first_violation is None:
                        first_violation = (
                            f"Chunk boundary: "
                            f"{previous_timestamp} -> "
                            f"{valid.iloc[0]}"
                        )

            if current_reversals > 0:
                reversal_count += current_reversals

                if first_violation is None:
                    position = inversions[inversions].index[0]
                    first_violation = (
                        f"Within chunk near index {position}"
                    )

            previous_timestamp = valid.iloc[-1]

        results.append({
            "class": row["scenario_label"],
            "file": filename,
            "headerless": filename in headerless_files,
            "records": records,
            "valid_timestamps": valid_count,
            "invalid_timestamps": invalid_count,
            "reversal_count": reversal_count,
            "chronologically_sorted": reversal_count == 0,
            "earliest_timestamp": earliest,
            "latest_timestamp": latest,
            "first_violation": first_violation
        })

    except Exception as e:

        results.append({
            "class": row["scenario_label"],
            "file": filename,
            "headerless": filename in headerless_files,
            "error": str(e)
        })

    if (i + 1) % 10 == 0 or i + 1 == len(manifest):
        print(f"Processed {i + 1}/{len(manifest)} files")

# ------------------------------------------------------------
# 5. SAVE RESULTS
# ------------------------------------------------------------

order_df = pd.DataFrame(results)

output_path = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_corrected_timestamp_order_audit.csv"
)

order_df.to_csv(output_path, index=False)

# ------------------------------------------------------------
# 6. SUMMARY
# ------------------------------------------------------------

successful = order_df[
    order_df["error"].isna()
].copy() if "error" in order_df.columns else order_df.copy()

print("\n" + "=" * 75)
print("CORRECTED DIAD TIMESTAMP ORDER AUDIT")
print("=" * 75)

print("Total files:", len(order_df))
print("Successful files:", len(successful))
print("Files with errors:", len(order_df) - len(successful))

print("Total records:", successful["records"].sum())
print("Valid timestamps:", successful["valid_timestamps"].sum())
print("Invalid timestamps:", successful["invalid_timestamps"].sum())

print(
    "Total timestamp reversals:",
    successful["reversal_count"].sum()
)

print(
    "Chronologically sorted files:",
    successful["chronologically_sorted"].sum()
)

print(
    "Files with timestamp reversals:",
    (~successful["chronologically_sorted"]).sum()
)

print("\nHEADERLESS FILE RESULTS:")

print(
    successful[
        successful["headerless"]
    ][
        ["file", "records", "valid_timestamps",
         "invalid_timestamps", "reversal_count"]
    ].to_string(index=False)
)

print("\nSaved:", output_path)


Reference columns: 84
Processed 10/126 files
Processed 20/126 files
Processed 30/126 files
Processed 40/126 files
Processed 50/126 files
Processed 60/126 files
Processed 70/126 files
Processed 80/126 files
Processed 90/126 files
Processed 100/126 files
Processed 110/126 files
Processed 120/126 files
Processed 126/126 files

CORRECTED DIAD TIMESTAMP ORDER AUDIT
Total files: 126
Successful files: 126
Files with errors: 0
Total records: 26969082
Valid timestamps: 26969082
Invalid timestamps: 0
Total timestamp reversals: 4063873
Chronologically sorted files: 5
Files with timestamp reversals: 121

HEADERLESS FILE RESULTS:
                         file  records  valid_timestamps  invalid_timestamps  reversal_count
  DoS-TCP_Flood.pcap_Flow.csv        1                 1                   0               0
 DoS-TCP_Flood1.pcap_Flow.csv        1                 1                   0               0
DoS-TCP_Flood10.pcap_Flow.csv        1                 1                   0               0
 Do

In [44]:

# ============================================================
# O1: VERIFY FIVE HEADERLESS DIAD FILES
# ============================================================

import os
import pandas as pd

OUTPUT_DIR = "/kaggle/working/O1_results"

manifest = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_scenario_label_mapping_corrected.csv"
    )
)

headerless_files = [
    "DoS-TCP_Flood.pcap_Flow.csv",
    "DoS-TCP_Flood1.pcap_Flow.csv",
    "DoS-TCP_Flood2.pcap_Flow.csv",
    "DoS-TCP_Flood3.pcap_Flow.csv",
    "DoS-TCP_Flood10.pcap_Flow.csv"
]

reference_path = manifest.loc[
    manifest["file_name"] == "BenignTraffic.pcap_Flow.csv",
    "file_path"
].iloc[0]

reference_columns = pd.read_csv(
    reference_path,
    nrows=0
).columns.tolist()

timestamp_position = reference_columns.index("Timestamp")

print("Reference columns:", len(reference_columns))
print("Timestamp position:", timestamp_position)

print("\n" + "=" * 80)
print("HEADERLESS FILE STRUCTURAL VERIFICATION")
print("=" * 80)

for filename in headerless_files:

    path = manifest.loc[
        manifest["file_name"] == filename,
        "file_path"
    ].iloc[0]

    print("\n" + "-" * 80)
    print("FILE:", filename)
    print("SIZE:", os.path.getsize(path), "bytes")

    # Inspect raw lines
    with open(path, "r", encoding="utf-8-sig",
              errors="replace") as f:

        raw_lines = [f.readline().rstrip("\n") for _ in range(3)]

    for j, line in enumerate(raw_lines, start=1):
        print(f"RAW LINE {j}:", line[:250])

    # Inspect CSV without assuming headers
    try:

        sample = pd.read_csv(
            path,
            header=None,
            nrows=3,
            low_memory=False
        )

        print("Detected columns:", sample.shape[1])
        print("Sample rows:", len(sample))

        if sample.shape[1] == len(reference_columns):

            print("SCHEMA WIDTH: MATCH")

            print(
                "First three timestamp values:",
                sample.iloc[:, timestamp_position].tolist()
            )

        else:

            print("SCHEMA WIDTH: MISMATCH")

    except Exception as e:
        print("READ ERROR:", str(e))

    # Count actual data rows without loading the entire file
    try:

        total_rows = sum(
            1 for _ in open(
                path,
                "r",
                encoding="utf-8-sig",
                errors="replace"
            )
        )

        print("Physical lines in file:", total_rows)

    except Exception as e:
        print("LINE COUNT ERROR:", str(e))

print("\n" + "=" * 80)
print("STRUCTURAL VERIFICATION COMPLETED")
print("=" * 80)


Reference columns: 84
Timestamp position: 6

HEADERLESS FILE STRUCTURAL VERIFICATION

--------------------------------------------------------------------------------
FILE: DoS-TCP_Flood.pcap_Flow.csv
SIZE: 460 bytes
RAW LINE 1: 35.185.101.66-192.168.137.72-443-33777-6,35.185.101.66,443,192.168.137.72,33777,6,22/07/2022 11:41:31 AM,9592154,51,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0.0,5.316845413449367,191843.07999999996,219603.38466303944,1074112.0,33354.0,9592154.0,191843.07999
RAW LINE 2: 
RAW LINE 3: 
Detected columns: 84
Sample rows: 1
SCHEMA WIDTH: MATCH
First three timestamp values: ['22/07/2022 11:41:31 AM']
Physical lines in file: 1

--------------------------------------------------------------------------------
FILE: DoS-TCP_Flood1.pcap_Flow.csv
SIZE: 558 bytes
RAW LINE 1: 192.168.137.164-192.168.137.39-80-34050-6,192.168.137.164,80,192.168.137.39,34050,6,22/07/2022 04:00:53 PM,441263,7,1,1526.0,0.0,763.0,0.0,218.0,372.3058778298654,0.0,0.0,0.0,0.0,3458.2550542420277,18.129777479

In [45]:

import os
import pandas as pd

OUTPUT_DIR = "/kaggle/working/O1_results"
MANIFEST_PATH = os.path.join(
    OUTPUT_DIR,
    "O1_DIAD_scenario_label_mapping_corrected.csv"
)

manifest = pd.read_csv(MANIFEST_PATH)

headerless_files = {
    "DoS-TCP_Flood.pcap_Flow.csv",
    "DoS-TCP_Flood1.pcap_Flow.csv",
    "DoS-TCP_Flood2.pcap_Flow.csv",
    "DoS-TCP_Flood3.pcap_Flow.csv",
    "DoS-TCP_Flood10.pcap_Flow.csv"
}

scenario_counts = {}
file_results = []
CHUNK_SIZE = 200000

for i, row in manifest.iterrows():
    path = row["file_path"]
    filename = row["file_name"]
    scenario = str(row["scenario_label"])

    header = None if filename in headerless_files else 0
    record_count = 0

    reader = pd.read_csv(
        path,
        header=header,
        usecols=[0],
        chunksize=CHUNK_SIZE,
        encoding="utf-8-sig"
    )

    for chunk in reader:
        record_count += len(chunk)

    scenario_counts[scenario] = (
        scenario_counts.get(scenario, 0) + record_count
    )

    file_results.append({
        "file_name": filename,
        "scenario_label": scenario,
        "records": record_count
    })

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(manifest)} files")

scenario_df = pd.DataFrame(
    list(scenario_counts.items()),
    columns=["Scenario", "Records"]
)

scenario_df["Percentage"] = (
    scenario_df["Records"] /
    scenario_df["Records"].sum() * 100
)

scenario_df = scenario_df.sort_values(
    "Records", ascending=False
)

print("\n" + "=" * 70)
print("DIAD COMPLETE SCENARIO RECORD DISTRIBUTION")
print("=" * 70)
print(scenario_df.to_string(index=False))

print("\nTotal files:", len(file_results))
print("Total records:", sum(scenario_counts.values()))

print("\nExpected timestamp-audit records: 26,969,082")
print(
    "Difference:",
    sum(scenario_counts.values()) - 26969082
)

scenario_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_complete_scenario_record_distribution.csv"
    ),
    index=False
)

pd.DataFrame(file_results).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "O1_DIAD_record_count_by_file.csv"
    ),
    index=False
)

print("\nBoth reports saved in:", OUTPUT_DIR)


Processed 10/126 files
Processed 20/126 files
Processed 30/126 files
Processed 40/126 files
Processed 50/126 files
Processed 60/126 files
Processed 70/126 files
Processed 80/126 files
Processed 90/126 files
Processed 100/126 files
Processed 110/126 files
Processed 120/126 files

DIAD COMPLETE SCENARIO RECORD DISTRIBUTION
   Scenario  Records  Percentage
        DoS 23143929   85.816525
       DDoS  2637892    9.781171
      Recon   442158    1.639500
     Benign   398330    1.476988
      Mirai   174588    0.647364
   Spoofing   157238    0.583031
  Web-Based    11328    0.042004
Brute Force     3619    0.013419

Total files: 126
Total records: 26969082

Expected timestamp-audit records: 26,969,082
Difference: 0

Both reports saved in: /kaggle/working/O1_results
